# Paso 3 — Aplicación de tratamientos y consolidación

**Proyecto:** seguimiento de residentes del Puericultorio Pérez Araníbar.

En el paso 2 se identificaron las inconsistencias, aquí se aplican los tratamientos correspondientes y se arma el consolidado `Dataset_unificado.xlsx`.

| # | Tratamiento | Criterio |
|---|---|---|
| 1 | **Excluir los códigos (IDs) inconsistentes** | Se quitan todas sus fichas nominales y todas sus evaluaciones de nutrición. |
| 2 | **Excluir las evaluaciones con `PROVINCIA`** | En ese mes esa persona no tuvo evaluación, su ficha nominal se conserva. |
| 3 | **Normalizar el texto de los diagnósticos** | Una sola escritura por categoría (mayúsculas, espacios, errores). |
| 4 | **Completar el IMC faltante** | IMC = peso (kg) ÷ talla² (m), con una columna indicadora TRUE/FALSE. |
| 5 | **Calcular los diagnósticos con asterisco (`*`) que corresponden** | Con las tablas del INS, solo donde el indicador aplica por edad, con una columna indicadora TRUE/FALSE por cada diagnóstico. |

Después se unen la ficha nominal y la nutrición ya tratadas, se verifica el resultado y se guarda.

**Evidencia:** cada separación de datos se guarda como un Excel para presentarla a la entidad correspondiente. Los consolidados finales quedan en otro excel.

| # | Tratamiento | Criterio |
|---|---|---|
| 6 | **IMC con diferencia mayor a 0.11: se conserva el calculado** | Solo si el IMC se pudo calcular (peso ÷ talla²). Se guarda la fila completa de cada NNA modificado, antes y después. |
| 7 | **Limpiar y homologar la ficha nominal** | Sin comillas, fechas reales y una sola escritura por valor.
| 8 | **Evaluar los vacíos para el Paso 4** | Por columna: conservar, completar o eliminar.

In [ ]:
# LIBRERIAS
from pathlib import Path
from collections import Counter, defaultdict
from datetime import date, datetime
import hashlib
import html
import json
import re
import numpy as np
import pandas as pd
import openpyxl
from openpyxl.utils import column_index_from_string, get_column_letter
import unicodedata
from IPython.display import display, HTML

**Configuración.** Se leen los mismos archivos de entrada que en el Paso 2 (`Data Ficha Nominal`, `Data Nutrición`) y, además, las tablas del INS transcritas a CSV (`INS/tablas_transcritas`). Se guarda la huella digital (SHA-256) de cada archivo de entrada: al final se comprueba que **ninguno cambió**, es decir, que todos los tratamientos se hicieron en memoria y nunca sobre los originales.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
project_dir = Path('/content/drive/MyDrive/TA - Análisis de Datos [1INF03]')
project_dir

PosixPath('/content/drive/MyDrive/TA - Análisis de Datos [1INF03]')

In [ ]:
# RUTAS DEL PROYECTO (ejecución local: el notebook vive en la carpeta "Notebooks")

nominal_dir = project_dir / "B - Data" / "A - Source"     # Excel de la ficha nominal
nutrition_dir = project_dir / "B - Data" / "A - Source"       # Excel mensuales de nutrición

ins_tables_dir = project_dir / "TA" / "INS" / "tablas_transcritas"          # tablas INS transcritas a CSV
evidence_dir = project_dir / "C - KDD Steps" / "C - 3: Data Cleaning and Preprocessing"      # Excel de evidencia de cada separación de datos
consolidated_dir = project_dir      # consolidados finales
evidence_dir.mkdir(parents=True, exist_ok=True)
consolidated_dir.mkdir(parents=True, exist_ok=True)

# Archivos que devuelve este notebook
nutrition_path = consolidated_dir / "Nutricion_consolidada.xlsx"     # todo nutrición unido y tratado
unified_path = consolidated_dir / "Dataset_unificado.xlsx"           # ficha nominal + nutrición tratada

nominal_files = sorted(nominal_dir.glob("Ficha*.xlsx"))
nutrition_files = sorted(nutrition_dir.glob("Nutricion*.xlsx"))

assert len(nominal_files) == 1, "Se espera una ficha nominal consolidada."
assert len(nutrition_files) == 6, "Se esperan los seis archivos nutricionales de la entrega."

nominal_path = nominal_files[0]

# Huella digital de los archivos de entrada (permite comprobar que no se modificaron)
source_fingerprints = {
    path.name: hashlib.sha256(path.read_bytes()).hexdigest()
    for path in nominal_files + nutrition_files
}

pd.set_option("display.max_colwidth", 100)

# Para visualizar mejor las tablas
def show_table(table, title=None):
    if isinstance(table, pd.Series):
        table = table.to_frame()
    if title:
        display(HTML(f"<p><strong>{html.escape(title)}</strong></p>"))
    display(HTML('<div class="preparation-table">' +
                 table.to_html(index=True, escape=True, na_rep="—", border=0) + '</div>'))

# Guarda cada separación de datos en su propio Excel (evidencia para la entidad correspondiente)
def export_evidence(filename, sheets, folder=None):
    """Escribe un Excel con una hoja por tabla. `sheets` = {nombre_hoja: DataFrame}."""
    folder = folder or evidence_dir
    folder.mkdir(parents=True, exist_ok=True)
    path = folder / filename
    with pd.ExcelWriter(path, engine="openpyxl") as writer:
        for name, frame in sheets.items():
            frame = frame.copy()
            # Un texto que empieza con "=" se interpretaría como fórmula en Excel: se antepone una comilla
            for column in frame.columns:
                if frame[column].dtype == "object":
                    frame[column] = frame[column].map(
                        lambda v: "'" + v if isinstance(v, str) and v.startswith("=") else v)
            frame.to_excel(writer, sheet_name=name[:31], index=False)
    print(f"Evidencia guardada: {path.relative_to(project_dir)}  ({', '.join(f'{k}: {len(v)} filas' for k, v in sheets.items())})")

# Controla lo visual
display(HTML("""<style>
.preparation-table {overflow-x:auto; margin:12px 0 22px;}
.preparation-table table {border-collapse:collapse; font-size:13px; line-height:1.5;}
.preparation-table th {background:#edf2f6; color:#24384b; padding:9px; text-align:left;}
.preparation-table td {padding:8px 9px; text-align:left; vertical-align:top; border-bottom:1px solid #e4e9ee; min-width:75px; max-width:480px; overflow-wrap:anywhere; white-space:normal;}
.preparation-table tr:nth-child(even) {background:#f8fafc;}
</style>"""))

# 1. Lectura de los archivos recibidos

In [ ]:
# LECTURA DE LOS ARCHIVOS Y INVENTARIO

# Conservar las celdas nominales tal como llegaron, antes de preparar claves.
nominal_book = openpyxl.load_workbook(nominal_path, read_only=True, data_only=False)
nominal_sheet = nominal_book.worksheets[0]
nominal_sheet.reset_dimensions()
nominal_sheet_name = nominal_sheet.title
nominal_source_rows = list(nominal_sheet.iter_rows(values_only=True))
nominal_headers = list(nominal_source_rows[0])
nominal_book.close()

assert len(nominal_headers) == 33 # cantidad de columnas de ficha nominal
assert not any(isinstance(value, str) and value.startswith("=")
               for row in nominal_source_rows for value in row)

# En nutrición se leen tanto los resultados guardados como las fórmulas.
nutrition_sources = {}
for path in nutrition_files:
    values_book = openpyxl.load_workbook(path, data_only=True)
    formulas_book = openpyxl.load_workbook(path, data_only=False)
    sheets = []
    for sheet in values_book:
        sheets.append({
            "sheet": sheet.title,
            "rows": list(sheet.iter_rows(values_only=True)),
            "formulas": list(formulas_book[sheet.title].iter_rows(values_only=True)),
            "merged_ranges": [str(rng) for rng in sheet.merged_cells.ranges]
        })
    nutrition_sources[path.name] = sheets
    values_book.close()
    formulas_book.close()

file_inventory = [{
    "archivo": nominal_path.name, "fuente": "Nominal", "hojas": 1,
    "registros": len(nominal_source_rows) - 1,
    "organización": "Tabla nominal consolidada; encabezado en fila 1"
}]
for filename, sheets in nutrition_sources.items():
    file_inventory.append({
        "archivo": filename, "fuente": "Nutrición", "hojas": len(sheets),
        "registros": sum(bool(re.fullmatch(r"PPA_FN_\d+", str(row[1])))
                         for sheet in sheets for row in sheet["rows"] if len(row) > 1),
        "organización": "Hojas por hogar; encabezados y secciones por casa"
    })
file_inventory = pd.DataFrame(file_inventory).set_index("archivo")
show_table(file_inventory, "Inventario de la entrega")

/usr/local/lib/python3.13/dist-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)


,fuente,hojas,registros,organización
archivo,,,,
FichaNominal_consolidado_IDs_v2.xlsx,Nominal,1,2667,Tabla nominal consolidada; encabezado en fila 1
Nutricion_IDs_202601 - ENERO - TODOS.xlsx,Nutrición,5,141,Hojas por hogar; encabezados y secciones por casa
Nutricion_IDs_202602 - FEBRERO- INFANTES.xlsx,Nutrición,1,7,Hojas por hogar; encabezados y secciones por casa
Nutricion_IDs_202603 - MARZO- INFANTES.xlsx,Nutrición,1,6,Hojas por hogar; encabezados y secciones por casa
Nutricion_IDs_202604 - ABRIL - TODOS.xlsx,Nutrición,5,108,Hojas por hogar; encabezados y secciones por casa
Nutricion_IDs_202606 - JUNIO- TODOS.xlsx,Nutrición,5,108,Hojas por hogar; encabezados y secciones por casa
Nutricion_IDs_202608 - AGOSTO- OMG.xlsx,Nutrición,2,32,Hojas por hogar; encabezados y secciones por casa


In [ ]:
# Convertir la ficha nominal leída con openpyxl en un DataFrame
df_nominal = pd.DataFrame(
    nominal_source_rows[1:],
    columns=nominal_headers
)
print("Dimensiones de la ficha nominal:", df_nominal.shape)
display(df_nominal.head())

Dimensiones de la ficha nominal: (2667, 33)


,periodo,IDs_FichaNominal,Sexo,Fecha de nacimiento,Departamento,Provincia,Distrito,Nacionalidad,Fecha de ingreso,Observaciones y/o comentarios de la forma de ingreso de la NNA,...,Tipo de Institución Educativa,Acceso a capacitación técnico productiva,La NNA actualmente estudia,Estado de Salud Física Actual,Presenta Discapacidad (Indicar tipo de discapacidad),Nivel Nutricional,Soporte Psico-emocional,Atenciones de salud psicológica,Atención en Salud mental especializada,Seguro de Salud (NNA)
0,"""2025_01""",PPA_FN_0001,"""F""","""2022-03-26""","""LIMA""","""LIMA""","""LIMA""","""PERUANA""","""2022-09-06""","""Presunta desprotección familiar""",...,"""Educación Básica Regular -EBR""","""No aplica""","""Si""","""Saludable""","""No tiene""","""Sobrepeso""","""Acompañamiento/Estimulación""","""NO""","""No aplica""","""SIS"""
1,"""2025_01""",PPA_FN_0002,"""M""","""2021-11-30""","""LIMA""","""LIMA""","""CALLAO""","""PERUANO""","""13/12/2021""","""Presunta desprotección familiar""",...,"""Ninguno""","""No aplica""","""No""","""Saludable""","""No tiene""","""Saludable""","""Acompañamiento/Estimulación""","""NO""","""No aplica""","""SIS"""
2,"""2025_01""",PPA_FN_0003,"""M""","""2022-03-28""","""LIMA""","""LIMA""","""CALLAO""","""PERUANO""","""2022-04-04""","""Presunta desprotección familiar""",...,"""Ninguno""","""No aplica""","""No""","""Saludable""","""No tiene""","""Riesgo de baja talla""","""Acompañamiento/Estimulación""","""NO""","""No aplica""","""SIS"""
3,"""2025_01""",PPA_FN_0004,"""M""","""2023-09-01""","""LIMA""","""LIMA""","""LIMA""","""PERUANO""","""16/01/2023""","""Presunta desprotección familiar""",...,"""Ninguno""","""No aplica""","""No""","""Saludable""","""No tiene""","""Saludable""","""Estimulación temprana""","""NO""","""No aplicca""","""SIS"""
4,"""2025_01""",PPA_FN_0005,"""F""","""2021-11-06""","""LIMA""","""LIMA""","""CALLAO""","""PERUANA""","""15/11/2021""","""Presunta desprotección familiar""",...,"""Ninguno""","""No aplica""","""No""","""Saludable""","""No tiene""","""Riesgo de baja talla""","""Acompañamiento/Estimulación""","""NO""","""No aplica""","""SIS"""


**Comentario 1:** 2 667 fichas nominales (33 columnas) y 402 evaluaciones de nutrición en 19 hojas. Esta lectura es idéntica a la del Paso 2.

# 2. Identificación de los códigos (IDs) inconsistentes

Se repite la validación de identidad del Paso 2 (mismo código y mismos criterios) para obtener la lista de códigos que se deben excluir.

In [ ]:
# PREPARACIÓN DE CLAVES NOMINALES Y REVISIÓN DE IDENTIDADES

def clean_text(value):
    """Limpia un texto: None -> "", espacios múltiples -> uno solo, quita espacios y comillas de los bordes."""
    if value is None:
        return ""
    return re.sub(r"\s+", " ", str(value)).strip().strip('"').strip()

def serialize_original(value):
    # Convierte fechas a texto ISO (AAAA-MM-DD) para guardarlas
    return value.isoformat() if isinstance(value, (date, datetime)) else value

def parse_birth_date(value):
    # fecha de nacimiento como texto
    if isinstance(value, (date, datetime)):
        return value.strftime("%Y-%m-%d")
    for date_format in ("%Y-%m-%d", "%d/%m/%Y", "%d-%m-%Y"):
        try:
            return datetime.strptime(clean_text(value), date_format).strftime("%Y-%m-%d")
        except ValueError:
            continue
    return None

def numeric_value(value):
    # retorna el valor solo si es numero
    return value if isinstance(value, (int, float)) and not isinstance(value, bool) else None

# Funciones de normalización para los campos nuevos (vacío -> None)
def fecha_comparable(x):
    """Fecha en formato AAAA-MM-DD; vacío -> None."""
    if pd.isna(x):
        return None
    return parse_birth_date(x) or (clean_text(x) or None)


def texto_comparable(x):
    """Texto en minúsculas sin espacios extra; vacío -> None."""
    if pd.isna(x):
        return None
    return clean_text(x).casefold() or None

In [ ]:
# SELECCIÓN DE CAMPOS PARA VALIDAR LA IDENTIDAD DE LOS NNA
# Posiciones (desde 0): 0 periodo, 1 id, 2 sexo, 3 fecha de nacimiento,
# 7 nacionalidad, 8 fecha de ingreso, 9 observaciones forma de ingreso, 14 fecha del oficio de ingreso

columnas = [0, 1, 2, 3, 7, 8, 9, 14]
revision = df_nominal.iloc[:, columnas].copy()
print("Columnas seleccionadas:", list(revision.columns))   # Verifica que sean las correctas
revision.columns = ["periodo", "id_persona", "sexo", "fecha_nacimiento",
                    "nacionalidad", "fecha_ingreso", "obs_forma_ingreso", "fecha_oficio_ingreso"]


# Normalizar los valores utilizando las funciones originales
revision["periodo"] = revision["periodo"].apply(
    lambda x: clean_text(x).replace("_", "-")
)
revision["id_persona"] = revision["id_persona"].apply(clean_text)
revision["sexo"] = revision["sexo"].apply(clean_text)
revision["fecha_nacimiento"] = revision["fecha_nacimiento"].apply(parse_birth_date)
revision["nacionalidad"] = revision["nacionalidad"].apply(texto_comparable)
revision["fecha_ingreso"] = revision["fecha_ingreso"].apply(fecha_comparable)
revision["obs_forma_ingreso"] = revision["obs_forma_ingreso"].apply(texto_comparable)
revision["fecha_oficio_ingreso"] = revision["fecha_oficio_ingreso"].apply(fecha_comparable)


# AGRUPAR POR NNA Y EVALUAR LA CONSISTENCIA
resultado = revision.groupby("id_persona").agg(
    registros=("periodo", "size"),
    nacimientos_distintos=("fecha_nacimiento", lambda x: x.nunique(dropna=False)),
    sexos_distintos=("sexo", "nunique"),
    nacionalidad_distintas=("nacionalidad", "nunique"),
    fecha_ingreso_distintas=("fecha_ingreso", "nunique"),
    fecha_oficio_distintas=("fecha_oficio_ingreso", "nunique"),
    obs_forma_ingreso_distintas=("obs_forma_ingreso", "nunique")
)

# CONTAR LOS MESES CON MÁS DE UNA FICHA
duplicados = (
    revision.groupby(["id_persona", "periodo"])
    .size()
    .gt(1)
    .groupby(level=0)
    .sum()
)
resultado["meses_con_varias_fichas"] = duplicados


# IDENTIFICAR LOS CÓDIGOS CON INCONSISTENCIAS
inconsistentes = resultado[
    (resultado["nacimientos_distintos"] > 1) |
    (resultado["sexos_distintos"] > 1) |
    (resultado["meses_con_varias_fichas"] > 0) |
    (resultado["nacionalidad_distintas"] > 1) |
    (resultado["fecha_ingreso_distintas"] > 1) |
    (resultado["fecha_oficio_distintas"] > 1) |
    (resultado["obs_forma_ingreso_distintas"] > 1)
]


# Mostrar los resultados
print("Identificadores que requieren revisión:", len(inconsistentes))
display(inconsistentes)

Columnas seleccionadas: ['periodo', 'IDs_FichaNominal', 'Sexo', 'Fecha de nacimiento', 'Nacionalidad', 'Fecha de ingreso', 'Observaciones y/o comentarios de la forma de ingreso de la NNA', 'Fecha del Oficio de Ingreso']
Identificadores que requieren revisión: 8


,registros,nacimientos_distintos,sexos_distintos,nacionalidad_distintas,fecha_ingreso_distintas,fecha_oficio_distintas,obs_forma_ingreso_distintas,meses_con_varias_fichas
id_persona,,,,,,,,
PPA_FN_0010,19,2,1,1,1,1,1,0
PPA_FN_0020,36,5,2,2,5,5,1,9
PPA_FN_0025,19,2,1,1,1,1,1,0
PPA_FN_0035,19,1,1,1,2,1,1,0
PPA_FN_0036,19,1,1,1,2,1,1,0
PPA_FN_0071,19,1,1,1,2,1,1,0
PPA_FN_0082,19,1,1,1,2,2,1,0
PPA_FN_0127,19,2,1,1,1,1,1,0


In [ ]:
# Obtener los identificadores inconsistentes detectados previamente
conflicting_ids = set(inconsistentes.index)

print("Identificadores que requieren revisión:")
print(sorted(conflicting_ids))

Identificadores que requieren revisión:
['PPA_FN_0010', 'PPA_FN_0020', 'PPA_FN_0025', 'PPA_FN_0035', 'PPA_FN_0036', 'PPA_FN_0071', 'PPA_FN_0082', 'PPA_FN_0127']


**Comentario 2:** PPA_FN_0010, 0020, 0025, 0035, 0036, 0071, 0082 y 0127 son los mismos del Paso 2, cambian de fecha de nacimiento, sexo, nacionalidad, fecha de ingreso u oficio entre meses, o tienen más de una ficha por mes. Con estos datos no se puede garantizar que un mismo código corresponda siempre a la misma persona, y por eso no se usan para formar el consolidado.

In [ ]:
# TRATAMIENTO 1 (lado nominal): SEPARAR LAS FICHAS DE LOS CÓDIGOS INCONSISTENTES
# Estos códigos no se usarán para formar el consolidado. Se guardan aparte, completos, como evidencia.
mask_nominal_excluida = df_nominal.iloc[:, 1].map(clean_text).isin(conflicting_ids)

nominal_excluida = df_nominal[mask_nominal_excluida].copy()
nominal_excluida.insert(0, "fila_excel_origen", nominal_excluida.index + 2)
nominal_excluida = nominal_excluida.sort_values([df_nominal.columns[1], df_nominal.columns[0]])

print("Fichas nominales de la ficha completa:", len(df_nominal))
print("Fichas separadas por código inconsistente:", len(nominal_excluida))
print("Fichas que quedan para el consolidado:", len(df_nominal) - len(nominal_excluida))

resumen_ids_excluidos = inconsistentes.copy()
resumen_ids_excluidos["fichas_nominales_excluidas"] = nominal_excluida.groupby(df_nominal.columns[1]).size()
show_table(resumen_ids_excluidos, "Códigos excluidos del consolidado")

Fichas nominales de la ficha completa: 2667
Fichas separadas por código inconsistente: 169
Fichas que quedan para el consolidado: 2498


,registros,nacimientos_distintos,sexos_distintos,nacionalidad_distintas,fecha_ingreso_distintas,fecha_oficio_distintas,obs_forma_ingreso_distintas,meses_con_varias_fichas,fichas_nominales_excluidas
id_persona,,,,,,,,,
PPA_FN_0010,19,2,1,1,1,1,1,0,19
PPA_FN_0020,36,5,2,2,5,5,1,9,36
PPA_FN_0025,19,2,1,1,1,1,1,0,19
PPA_FN_0035,19,1,1,1,2,1,1,0,19
PPA_FN_0036,19,1,1,1,2,1,1,0,19
PPA_FN_0071,19,1,1,1,2,1,1,0,19
PPA_FN_0082,19,1,1,1,2,2,1,0,19
PPA_FN_0127,19,2,1,1,1,1,1,0,19


**Comentario 3:** De las 2 667 fichas, 169 pertenecen a códigos inconsistentes (19 por cada código, y 36 para PPA_FN_0020, que tiene fichas duplicadas) y 2 498 continúan. Las 169 se guardan completas, con sus 33 columnas y su número de fila original, en `01_ids_inconsistentes_excluidos.xlsx`, para que se pueda revisar y corregir.

# 3. Extracción de las evaluaciones de nutrición

Se aplica la misma extracción del Paso 2: se identifican las plantillas de cada hoja y se leen las filas con código `PPA_FN_…`, conservando el valor original de cada celda y su ubicación.

In [ ]:
def inspect_nutrition_layout(source):
    rows = source["rows"]
    headers = rows[2]
    groups = {}
    for merged_range in source["merged_ranges"]:
        min_col, min_row, max_col, max_row = openpyxl.utils.range_boundaries(merged_range)
        if min_row == 2:
            title = rows[1][min_col - 1]
            for column in range(min_col, max_col + 1):
                groups[column] = title
    person_rows = [row for row in rows if len(row) > 1 and
                   re.fullmatch(r"PPA_FN_\d+", str(row[1]))]
    columns = []
    for index, header in enumerate(headers, start=1):
        if header is not None:
            values = [row[index - 1] if index <= len(row) else None for row in person_rows]
            columns.append({
                "column": get_column_letter(index),
                "header": re.sub(r"\s+", " ", str(header)).strip(),
                "group": groups.get(index),
                "types": Counter(type(value).__name__ for value in values)
            })
    return {"sheet": source["sheet"], "records": len(person_rows), "columns": columns}

nutrition_layout = [
    {"file": filename, "sheets": [inspect_nutrition_layout(sheet) for sheet in sheets]}
    for filename, sheets in nutrition_sources.items()
]
layout_review = []
for book in nutrition_layout:
    period_digits = re.search(r"2026\d{2}", book["file"]).group()
    period = period_digits[:4] + "-" + period_digits[4:]
    for sheet in book["sheets"]:
        layout_review.append({
            "periodo": period, "hoja": sheet["sheet"], "registros": sheet["records"],
            "encabezados": "; ".join(f'{c["column"]}: {c["header"]}' for c in sheet["columns"])
        })
show_table(pd.DataFrame(layout_review), "Plantillas nutricionales observadas")

,periodo,hoja,registros,encabezados
0,2026-01,INFANTES,33,A: N°; B: APELLIDOS Y NOMBRES; C: FECHA DE NACIMIENTO; D: EDAD; E: PESO (kg); F: TALLA (cm); G: IMC; H: P/E; I: T/E; J: P/T; K: IMC; L: P/E; M: T/E; N: P/T; O: IMC
1,2026-01,MIGUEL ECHENIQUE,17,A: N°; B: APELLIDOS Y NOMBRES; C: FECHA DE NACIMIENTO; D: EDAD; E: PESO (kg); F: TALLA (cm); G: IMC; H: T/E; I: IMC/E; J: T/E; K: IMC/E
2,2026-01,TOMAS VALLE,22,A: N°; B: APELLIDOS Y NOMBRES; C: FECHA DE NACIMIENTO; D: EDAD; E: PESO (kg); F: TALLA (cm); G: IMC; H: T/E; I: IMC/E; J: T/E; K: IMC/E
3,2026-01,MATOGROSO MUJERES,32,A: N°; B: APELLIDOS Y NOMBRES; C: FECHA DE NACIMIENTO; D: EDAD; E: PESO (kg); F: TALLA (cm); G: IMC; H: T/E; I: IMC/E; J: T/E; K: IMC
4,2026-01,MATOGROSO HOMBRES,37,A: N; B: APELLIDOS Y NOMBRES; C: FECHA DE NACIMIENTO; D: EDAD; E: PESO (kg); F: TALLA (cm); G: IMC; H: T/E; I: IMC/E; J: T/E; K: IMC
5,2026-02,Eva.Nut febrero,7,A: N; B: APELLIDOS Y NOMBRES; C: FECHA DE NACIMIENTO; D: EDAD; E: PESO (kg); F: TALLA (cm); G: IMC; H: P/E; I: T/E; J: P/T; K: IMC; L: P/E; M: T/E; N: P/T; O: IMC
6,2026-03,Eva.Nut marzo,6,A: N; B: APELLIDOS Y NOMBRES; C: FECHA DE NACIMIENTO; D: EDAD; E: PESO (kg); F: TALLA (cm); G: P.CEFALICO; H: P/E; I: T/E; J: P/T; K: IMC; L: P/E; M: T/E; N: P/T; O: IMC
7,2026-04,INFANTES (ABR),30,A: N; B: APELLIDOS Y NOMBRES; C: FECHA DE NACIMIENTO; D: EDAD; E: PESO (kg); F: TALLA (cm); G: IMC; H: P/E; I: T/E; J: P/T; K: IMC; L: P/E; M: T/E; N: P/T; O: IMC
8,2026-04,MIGUEL ECHENIQUE,16,A: N°; B: APELLIDOS Y NOMBRES; C: FECHA DE NACIMIENTO; D: EDAD; E: PESO (kg); F: TALLA (cm); G: PA; H: IMC; I: T/E; J: IMC/E; K: T/E; L: IMC/E
9,2026-04,TOMAS VALLE,20,A: N°; B: APELLIDOS Y NOMBRES; C: FECHA DE NACIMIENTO; D: EDAD; E: PESO (kg); F: TALLA (cm); G: P.A; H: IMC; I: P/E; J: T/E; K: IMC/E; L: T/E; M: IMC/E


In [ ]:
# DICCIONARIO que establece que valores incorrectos deben reemplazarse por su forma estandarizada

text_corrections = {
    "NORNAL": "NORMAL", "OBEESIDAD": "OBESIDAD", "OBESIDA": "OBESIDAD",
    "RIESO TALLA BAJA": "RIESGO TALLA BAJA", "RIESGO TALA BAJA": "RIESGO TALLA BAJA",
    "RIESGO DE TALLA BAJA": "RIESGO TALLA BAJA",
    "RIESGO DE SOBREPESO": "RIESGO SOBREPESO",
    "RIESGO SOBRE PESO": "RIESGO SOBREPESO", "SOBRE PESO": "SOBREPESO",
    "RIEGO DE DELGADEZ O RIESGO DE BAJO PESO": "RIESGO DE DELGADEZ O RIESGO DE BAJO PESO"
}

# Si existe el valor del diccionario en la data real, lo reemplaza por el valor correcto
def normalize_diagnosis(value):
    text = clean_text(value).upper()
    return text_corrections.get(text, text) if text not in ("", "*") else None

show_table(pd.DataFrame(text_corrections.items(), columns=["texto", "normalización"]).set_index("texto"),
           "Correcciones textuales explícitas")

,normalización
texto,
NORNAL,NORMAL
OBEESIDAD,OBESIDAD
OBESIDA,OBESIDAD
RIESO TALLA BAJA,RIESGO TALLA BAJA
RIESGO TALA BAJA,RIESGO TALLA BAJA
RIESGO DE TALLA BAJA,RIESGO TALLA BAJA
RIESGO DE SOBREPESO,RIESGO SOBREPESO
RIESGO SOBRE PESO,RIESGO SOBREPESO
SOBRE PESO,SOBREPESO


**Comentario 4:** Esta tabla lista los errores de escritura conocidos y su forma correcta (por ejemplo `NORNAL` → `NORMAL`, `OBEESIDAD` → `OBESIDAD`, `RIESGO DE SOBREPESO` → `RIESGO SOBREPESO`). Se aplican a todos los diagnósticos junto con el paso a mayúsculas y la eliminación de espacios y saltos de línea.

In [ ]:
# AMPLIACIÓN DE LAS CORRECCIONES DE TEXTO
# En el Paso 2 se vio que la talla alta aparece como "ALTO" (mayoría) y como "ALTA" (variante de género). Se unifica en "ALTO",
# que es la forma usada en la tabla de diagnóstico rápido del INS dentro de estos archivos.

text_corrections["ALTA"] = "ALTO"
show_table(pd.DataFrame(text_corrections.items(), columns=["texto", "normalización"]).set_index("texto"),
           "Correcciones textuales que se aplicarán (incluye la ampliación)")

,normalización
texto,
NORNAL,NORMAL
OBEESIDAD,OBESIDAD
OBESIDA,OBESIDAD
RIESO TALLA BAJA,RIESGO TALLA BAJA
RIESGO TALA BAJA,RIESGO TALLA BAJA
RIESGO DE TALLA BAJA,RIESGO TALLA BAJA
RIESGO DE SOBREPESO,RIESGO SOBREPESO
RIESGO SOBRE PESO,RIESGO SOBREPESO
SOBRE PESO,SOBREPESO


**Comentario 5:** En el Paso 2 se vio que la categoría de talla alta aparece como `ALTO` en la gran mayoría de las celdas y como `ALTA` en 2. Es la misma categoría escrita con otro género, y dejarla separada partiría el conteo. Es la **única ampliación** a las correcciones existentes. Las categorías que difieren por su significado (por ejemplo `BAJO PESO` frente a `DESNUTRIDO`) no se tocan: no son errores de escritura.

PREPARACIÓN DE ESTRUCTURA PARA ALMACENAR LOS DATOS PROCESADOS Y REGISTRAR LOS PROBLEMAS ENCONTRADOS

In [ ]:
records = []
issues = []
notes = []
normalization_counts = Counter()
def issue(rec, field, kind, detail, cell=None):
    issues.append({'id_evaluacion':rec['id_evaluacion'],'campo':field,'observacion':kind,'detalle':detail,
        'archivo_origen':rec['archivo_origen'],'hoja_origen':rec['hoja_origen'],'celda_origen':cell or str(rec['fila_origen'])})

In [ ]:
# DICCIONARIO

from collections import defaultdict

# Preparar las claves nominales desde el DataFrame
nominal_by_key = defaultdict(list)
nominal_by_id = defaultdict(list)

for fila, row in df_nominal.iterrows():

    periodo = clean_text(row.iloc[0]).replace("_", "-")
    ident = clean_text(row.iloc[1])

    registro = {
        "fila": fila + 2,
        "periodo": periodo,
        "id": ident,
        "sexo": clean_text(row.iloc[2]),
        "dob": parse_birth_date(row.iloc[3])
    }

    nominal_by_key[(periodo, ident)].append(registro)
    nominal_by_id[ident].append(registro)

# Identificar los meses disponibles en la ficha nominal
nominal_periods = {
    clean_text(valor).replace("_", "-")
    for valor in df_nominal.iloc[:, 0]
}

print("Claves nominales preparadas desde df.")
print("Identificadores:", len(nominal_by_id))
print("Periodos:", len(nominal_periods))

Claves nominales preparadas desde df.
Identificadores: 157
Periodos: 19


In [ ]:
# EXTRACCIÓN DE EVALUACIONES NUTRICIONALES

for f in nutrition_layout:

    path = nutrition_dir / f["file"]

    # Obtener el periodo desde el nombre del archivo
    period = re.search(r"2026\d{2}", path.name).group()
    period = period[:4] + "-" + period[4:]

    for s in f["sheets"]:

        # Obtener los datos originales de la hoja
        source = next(
            sheet for sheet in nutrition_sources[path.name]
            if sheet["sheet"] == s["sheet"]
        )

        rows = source["rows"]
        formulas = source["formulas"]
        hdr = rows[2]

        # Identificar las columnas de peso y talla
        wc = next(i for i, h in enumerate(hdr)
                  if h and "PESO" in str(h))
        hc = next(i for i, h in enumerate(hdr)
                  if h and "TALLA" in str(h))

        # Identificar las columnas de diagnósticos nutricionales
        diag = [
            c for c in s["columns"]
            if c["header"] in ("P/E", "T/E", "P/T", "IMC/E", "IMC")
            and c["group"] != "DATOS"
            and c["types"].get("str", 0) > 0
        ]

        # Clasificar diagnósticos según curva o tabla
        blockmap = {}
        seen = Counter()

        for c in diag:
            indicator = {
                "P/E": "pe",
                "T/E": "te",
                "P/T": "pt",
                "IMC/E": "imc",
                "IMC": "imc"
            }[c["header"]]

            seen[indicator] += 1
            block = "curva" if seen[indicator] == 1 else "tabla"

            if c["group"] and (
                "TABLA" in clean_text(c["group"]).upper()
                or "RAPIDO" in clean_text(c["group"]).upper()
            ):
                block = "tabla"

            blockmap[(indicator, block)] = c

        # Identificar columnas opcionales según la plantilla
        bmi = next(
            (column_index_from_string(c["column"]) - 1
             for c in s["columns"]
             if c["header"] == "IMC" and c not in diag),
            None
        )

        pc = next(
            (column_index_from_string(c["column"]) - 1
             for c in s["columns"]
             if c["header"] == "P.CEFALICO"),
            None
        )

        pa = next(
            (column_index_from_string(c["column"]) - 1
             for c in s["columns"]
             if c["header"] in ("PA", "P.A")),
            None
        )

        # Identificar el grupo y la casa
        group = (
            "INFANTES"
            if "INFANTES" in s["sheet"].upper()
            or s["sheet"].startswith("Eva.Nut")
            else s["sheet"].strip()
        )

        house = None

        # Recorrer los registros de la hoja nutricional
        for rn, r in enumerate(rows, 1):

            get = lambda ix: (
                r[ix] if ix is not None and ix < len(r) else None
            )

            fg = lambda ix: (
                formulas[rn - 1][ix]
                if ix is not None and ix < len(formulas[rn - 1])
                else None
            )

            # Identificar la casa a la que pertenecen los registros
            if (
                rn > 3
                and isinstance(get(0), str)
                and clean_text(get(0)).upper().startswith("CASA")
            ):
                house = clean_text(get(0))
                continue

            # Conservar las filas sin identificador como notas
            if not re.fullmatch(r"PPA_FN_\d+", str(get(1))):
                if rn > 3 and any(v is not None for v in r):
                    notes.append({
                        "periodo": period,
                        "grupo_hogar": group,
                        "archivo_origen": path.name,
                        "hoja_origen": s["sheet"],
                        "fila_origen": rn,
                        "nota_original": " / ".join(
                            f"{get_column_letter(i + 1)}{rn}: {serialize_original(v)}"
                            for i, v in enumerate(r)
                            if v is not None
                        )
                    })
                continue

            # Identificación de la evaluación
            ident = get(1)
            eid = f'NUT_{period.replace("-", "")}_{ident}'

            # Obtener mediciones originales
            w = get(wc)
            h = get(hc)
            born = get(2)

            # Identificar la estructura especial de edad en agosto
            aug = clean_text(hdr[3]) == "EDAD C/ FORMULA"
            ageix = 4 if aug else 3
            ageformula = fg(3)

            isageformula = (
                isinstance(ageformula, str)
                and ageformula.startswith("=")
            )

            # Crear el registro nutricional
            rec = {
                "id_evaluacion": eid,
                "id_persona": ident,
                "periodo": period,
                "grupo_hogar": group,
                "casa": house,

                # Medidas nutricionales
                "peso_kg": numeric_value(w),
                "talla_cm": numeric_value(h),
                "imc_reportado": numeric_value(get(bmi)),
                "imc_calculado": None,
                "diferencia_imc": None,

                "estado_mediciones": (
                    "sin_medicion_provincia"
                    if clean_text(w) == "PROVINCIA"
                    else "peso_talla_registrados"
                    if numeric_value(w) is not None
                    and numeric_value(h) is not None
                    else "mediciones_incompletas"
                ),

                # Información de nacimiento y edad
                "fecha_nacimiento": parse_birth_date(born),
                "fecha_evaluacion": None,
                "edad_reportada": (
                    serialize_original(get(ageix))
                    if aug or not isageformula else None
                ),
                "edad_formula_guardada": (
                    serialize_original(get(3))
                    if isageformula or aug else None
                ),

                # Información nominal pendiente de vinculación
                "sexo_nominal": None,
                "estado_vinculo_nominal": None,

                # Métodos de evaluación nutricional
                "metodo_curva_original": next(
                    (c["group"] for (ind, bl), c in blockmap.items()
                     if bl == "curva" and c["group"]),
                    None
                ),

                "metodo_tabla_original": next(
                    (c["group"] for (ind, bl), c in blockmap.items()
                     if bl == "tabla" and c["group"]),
                    None
                ),

                "indicador_imc_curva_original": (
                    blockmap.get(("imc", "curva"), {}).get("header")
                ),
                "indicador_imc_tabla_original": (
                    blockmap.get(("imc", "tabla"), {}).get("header")
                ),

                # Valores originales para trazabilidad
                "peso_original": serialize_original(w),
                "talla_original": serialize_original(h),
                "imc_original": serialize_original(get(bmi)),
                "imc_formula_original": (
                    fg(bmi) if isinstance(fg(bmi), str)
                    and fg(bmi).startswith("=") else None
                ),

                "perimetro_cefalico_original": serialize_original(get(pc)),
                "perimetro_cefalico_unidad": (
                    "no_declarada" if pc is not None else None
                ),

                "pa_original": serialize_original(get(pa)),
                "pa_encabezado_original": (
                    hdr[pa] if pa is not None else None
                ),
                "pa_unidad": "no_declarada" if pa is not None else None,

                "fecha_nacimiento_original": serialize_original(born),
                "edad_formula_original": (
                    ageformula if isageformula else None
                ),
                "numero_original": serialize_original(get(0)),
                "observacion_original": (
                    str(w) if numeric_value(w) is None
                    and clean_text(w) else None
                ),

                # Procedencia del registro
                "archivo_origen": path.name,
                "hoja_origen": s["sheet"],
                "fila_origen": rn,

                "columnas_diagnostico_originales": "; ".join(
                    f'{i}_{b}:{c["column"]}'
                    for (i, b), c in blockmap.items()
                ),

                # Vinculación con la ficha nominal
                "sexo_archivo_origen": None,
                "sexo_hoja_origen": None,
                "sexo_fila_origen": None,
                "fecha_nacimiento_nominal": None,

                "coincidencias_nominal_mes": len(
                    nominal_by_key.get((period, ident), [])
                ),

                "identidad_nominal_observacion": None,

                # Indicadores de calidad
                "asteriscos_diagnostico": 0,
                "vacios_diagnostico": 0,
                "indicadores_ausentes": 0,
                "normalizaciones_texto": 0,
                "diferencias_entre_metodos": 0,
                "observaciones_calidad": None
            }

            # PROCESAMIENTO DE DIAGNÓSTICOS NUTRICIONALES

            for block in ("curva", "tabla"):
                for indicator in ("pe", "te", "pt", "imc"):

                    key = f"dx_{indicator}_{block}"
                    c = blockmap.get((indicator, block))

                    v = (
                        get(column_index_from_string(c["column"]) - 1)
                        if c else None
                    )

                    # Identificar el estado original del dato
                    state = (
                        "ausente_en_plantilla" if not c
                        else "vacio_original" if clean_text(v) == ""
                        else "asterisco_original" if clean_text(v) == "*"
                        else "categoria_registrada"
                    )

                    rec[key] = normalize_diagnosis(v)
                    rec[key + "_original"] = serialize_original(v)
                    rec[key + "_estado"] = state

                    if state == "asterisco_original":
                        rec["asteriscos_diagnostico"] += 1
                    elif state == "vacio_original":
                        rec["vacios_diagnostico"] += 1
                    elif state == "ausente_en_plantilla":
                        rec["indicadores_ausentes"] += 1

                    # Registrar normalizaciones realizadas
                    if v is not None and state == "categoria_registrada":
                        normalization_counts[
                            (str(v), normalize_diagnosis(v))
                        ] += 1

                        if str(v) != normalize_diagnosis(v):
                            rec["normalizaciones_texto"] += 1

            # Comparar diagnósticos entre los dos métodos
            for indicator in ("pe", "te", "pt", "imc"):
                v1 = rec[f"dx_{indicator}_curva"]
                v2 = rec[f"dx_{indicator}_tabla"]

                if v1 is not None and v2 is not None and v1 != v2:
                    rec["diferencias_entre_metodos"] += 1

            # VINCULACIÓN CON LA FICHA NOMINAL

            # Buscar el identificador y periodo en la ficha nominal
            nr = nominal_by_key.get((period, ident), [])

            # NUEVO: aprovechar los identificadores inconsistentes
            # detectados previamente mediante pandas
            if ident in conflicting_ids:
                rec["identidad_nominal_observacion"] = (
                    "identificador_con_datos_contradictorios"
                )

            # Determinar si se puede realizar una unión segura
            if not nr:
                rec["estado_vinculo_nominal"] = (
                    "periodo_nominal_no_disponible"
                    if period not in nominal_periods
                    else "id_ausente_en_mes_nominal"
                )

            elif len(nr) > 1:
                rec["estado_vinculo_nominal"] = "multiples_coincidencias"

            elif nr[0]["dob"] != rec["fecha_nacimiento"]:
                rec["estado_vinculo_nominal"] = (
                    "fecha_nacimiento_no_coincide"
                )
                rec["fecha_nacimiento_nominal"] = nr[0]["dob"]

            elif rec["identidad_nominal_observacion"]:
                rec["estado_vinculo_nominal"] = (
                    "identidad_contradictoria_no_vinculada"
                )
                rec["fecha_nacimiento_nominal"] = nr[0]["dob"]

            else:
                # Unión segura: se obtiene el sexo de la ficha nominal
                rec.update(
                    sexo_nominal=nr[0]["sexo"],
                    estado_vinculo_nominal=(
                        "unico_mes_y_nacimiento_coinciden"
                    ),
                    sexo_archivo_origen=nominal_path.name,
                    sexo_hoja_origen=nominal_sheet_name,
                    sexo_fila_origen=nr[0]["fila"],
                    fecha_nacimiento_nominal=nr[0]["dob"]
                )

            # CONTROL DE CALIDAD DE LA EVALUACIÓN

            flags = []

            if rec["estado_mediciones"] != "peso_talla_registrados":
                flags.append("sin peso/talla: PROVINCIA")

            if rec["asteriscos_diagnostico"]:
                flags.append("asteriscos sin leyenda")

            if rec["vacios_diagnostico"]:
                flags.append("diagnosticos vacios")

            if rec["metodo_curva_original"] is None:
                flags.append("metodo curva sin titulo")

                issue(
                    rec,
                    "metodo_curva_original",
                    "titulo_no_declarado",
                    "El bloque 1 existe pero no declara metodo en su encabezado.",
                    "I2:J2"
                )

            if pa is not None:
                flags.append("PA: significado/unidad por confirmar")

            if isageformula and "TODAY" in ageformula.upper():
                flags.append("edad calculada con TODAY en origen")

            if rec["estado_vinculo_nominal"] != (
                "unico_mes_y_nacimiento_coinciden"
            ):
                flags.append("sexo nominal no vinculado")

                issue(
                    rec,
                    "sexo_nominal",
                    rec["estado_vinculo_nominal"],
                    "No se completa sexo mediante una union ambigua, "
                    "otro mes o fecha de nacimiento distinta."
                )

            # Verificar diferencias entre el IMC calculado y reportado
            calc = (
                numeric_value(w) / (numeric_value(h) / 100) ** 2
                if numeric_value(w) is not None
                and numeric_value(h) is not None
                and numeric_value(h) > 0
                else None
            )

            if (
                calc is not None
                and rec["imc_reportado"] is not None
                and abs(rec["imc_reportado"] - calc) > 0.11
            ):
                flags.append("IMC reportado difiere >0.11")

                issue(
                    rec,
                    "imc_reportado",
                    "diferencia_imc",
                    "Reportado menos peso/(talla/100)^2: "
                    + str(round(rec["imc_reportado"] - calc, 4)),
                    get_column_letter(bmi + 1) + str(rn)
                )

            # Guardar observaciones y evaluación nutricional
            rec["observaciones_calidad"] = "; ".join(flags) or None
            records.append(rec)

**Comentario 6:** Cada evaluación queda con el valor tratado (`dx_…`), el texto original (`dx_…_original`), el estado de la celda (categoría registrada, asterisco, vacía o ausente) y su ubicación exacta en el Excel (archivo, hoja, fila). Para demostrar más adelante qué se cambió y qué no.

CREACION DE TABLA CONSOLIDADA DE NUTRICION

In [ ]:
nutrition_df = pd.DataFrame(records)

IMC_REVIEW_TOLERANCE = 0.11 # Tolerencia del IMC que se calcula

# Validacion que cada registro tenga peso, talla y que la alla sea mayor que cero
valid_measurements = (nutrition_df["peso_kg"].notna() &
                      nutrition_df["talla_cm"].notna() &
                      (nutrition_df["talla_cm"] > 0))

# Si cumple lo anterior se calcula el IMC
nutrition_df.loc[valid_measurements, "imc_calculado"] = (
    nutrition_df.loc[valid_measurements, "peso_kg"] /
    (nutrition_df.loc[valid_measurements, "talla_cm"] / 100) ** 2
)


nutrition_df["imc_calculado"] = pd.to_numeric(nutrition_df["imc_calculado"], errors="raise")
nutrition_df["diferencia_imc"] = nutrition_df["imc_reportado"] - nutrition_df["imc_calculado"]

#Revision de IMC
nutrition_df["revision_imc"] = "no_comparable"
comparable = nutrition_df["diferencia_imc"].notna()
nutrition_df.loc[comparable, "revision_imc"] = "dentro_tolerancia"
nutrition_df.loc[comparable & (nutrition_df["diferencia_imc"].abs() > IMC_REVIEW_TOLERANCE),
                 "revision_imc"] = "revisar"

assert not nutrition_df["id_evaluacion"].duplicated().any()

# Mantener el mismo orden de columnas que el conjunto consolidado.
nutrition_columns = ['id_evaluacion', 'id_persona', 'periodo', 'grupo_hogar', 'casa', 'peso_kg', 'talla_cm', 'imc_reportado', 'imc_calculado', 'diferencia_imc', 'revision_imc', 'dx_pe_curva', 'dx_te_curva', 'dx_pt_curva', 'dx_imc_curva', 'dx_pe_tabla', 'dx_te_tabla', 'dx_pt_tabla', 'dx_imc_tabla', 'estado_mediciones', 'fecha_nacimiento', 'fecha_evaluacion', 'edad_reportada', 'edad_formula_guardada', 'sexo_nominal', 'estado_vinculo_nominal', 'metodo_curva_original', 'metodo_tabla_original', 'indicador_imc_curva_original', 'indicador_imc_tabla_original', 'peso_original', 'talla_original', 'imc_original', 'imc_formula_original', 'perimetro_cefalico_original', 'perimetro_cefalico_unidad', 'pa_original', 'pa_encabezado_original', 'pa_unidad', 'fecha_nacimiento_original', 'edad_formula_original', 'numero_original', 'observacion_original', 'archivo_origen', 'hoja_origen', 'fila_origen', 'columnas_diagnostico_originales', 'sexo_archivo_origen', 'sexo_hoja_origen', 'sexo_fila_origen', 'fecha_nacimiento_nominal', 'coincidencias_nominal_mes', 'identidad_nominal_observacion', 'asteriscos_diagnostico', 'vacios_diagnostico', 'indicadores_ausentes', 'normalizaciones_texto', 'diferencias_entre_metodos', 'observaciones_calidad', 'dx_pe_curva_original', 'dx_pe_curva_estado', 'dx_te_curva_original', 'dx_te_curva_estado', 'dx_pt_curva_original', 'dx_pt_curva_estado', 'dx_imc_curva_original', 'dx_imc_curva_estado', 'dx_pe_tabla_original', 'dx_pe_tabla_estado', 'dx_te_tabla_original', 'dx_te_tabla_estado', 'dx_pt_tabla_original', 'dx_pt_tabla_estado', 'dx_imc_tabla_original', 'dx_imc_tabla_estado']
nutrition_df = nutrition_df.reindex(columns=nutrition_columns)
normalization_summary = pd.DataFrame([
    {"original": original, "normalizado": normalized, "frecuencia": count}
    for (original, normalized), count in normalization_counts.items()
]).sort_values("frecuencia", ascending=False)

sheet_notes = pd.DataFrame(notes)
quality_issues = pd.DataFrame(issues)

show_table(pd.crosstab(nutrition_df["periodo"], nutrition_df["grupo_hogar"]),
           "Evaluaciones extraídas por mes y hogar")

grupo_hogar,INFANTES,MATOGROSO HOMBRES,MATOGROSO MUJERES,MIGUEL ECHENIQUE,TOMAS VALLE
periodo,,,,,
2026-01,33,37,32,17,22
2026-02,7,0,0,0,0
2026-03,6,0,0,0,0
2026-04,30,24,18,16,20
2026-06,30,22,18,17,21
2026-08,0,17,15,0,0


In [ ]:
show_table(nutrition_df["estado_mediciones"].value_counts().rename("evaluaciones"),
           "Disponibilidad de mediciones")

,evaluaciones
estado_mediciones,
peso_talla_registrados,379
sin_medicion_provincia,23


In [ ]:
# Para indicar cuantas veces se realizo cada correcion durante el preprocesamiento
show_table(normalization_summary[normalization_summary["original"] != normalization_summary["normalizado"]],
           "Cambios de texto observados en los diagnósticos")

show_table(nutrition_df["revision_imc"].value_counts().rename("evaluaciones"),
           "Revisión aritmética del IMC")

,original,normalizado,frecuencia
16,NORMAL,NORMAL,305
4,RIESGO TALLA BAJA,RIESGO TALLA BAJA,98
43,Normal,NORMAL,88
15,RIESGO SOBREPESO,RIESGO SOBREPESO,75
18,SOBREPESO,SOBREPESO,28
14,TALLA BAJA,TALLA BAJA,25
7,RIESGO SOBREPESO,RIESGO SOBREPESO,15
17,OBESIDAD,OBESIDAD,11
45,Riesgo de \ntalla baja,RIESGO TALLA BAJA,10
22,ALTO,ALTO,10


,evaluaciones
revision_imc,
dentro_tolerancia,364
no_comparable,29
revisar,9


**Comentario 7:** Todavía no se ha aplicado ninguna exclusión: son las mismas 402 evaluaciones del Paso 2 (379 con peso y talla, 23 con `PROVINCIA`), con 364 IMC dentro de la tolerancia, 29 no comparables y 9 con diferencia mayor a 0.11. La tabla de cambios de texto muestra los diagnósticos que difieren de su forma normalizada, y a partir de aquí empiezan los tratamientos.

In [ ]:
# Si la nutrición y la ficha nominal solo difieren en la fecha de nacimiento, se usa la de la ficha nominal
# La fecha que venía en nutrición queda en fecha_nacimiento_original, la evaluación se vincula y se anota en observaciones_calidad.

mask_nac = (nutrition_df["estado_vinculo_nominal"] == "fecha_nacimiento_no_coincide") & ~nutrition_df["id_persona"].isin(conflicting_ids)
filas_nac = []
for indice in nutrition_df.index[mask_nac]:
    r = nutrition_df.loc[indice]
    nr = nominal_by_key[(r["periodo"], r["id_persona"])]
    assert len(nr) == 1
    filas_nac.append({"id_evaluacion": r["id_evaluacion"], "id_persona": r["id_persona"], "periodo": r["periodo"],
                      "fecha_nacimiento_nutricion": r["fecha_nacimiento"], "fecha_nacimiento_ficha_nominal": nr[0]["dob"],
                      "archivo_origen": r["archivo_origen"], "hoja_origen": r["hoja_origen"], "fila_origen": r["fila_origen"]})
    nutrition_df.at[indice, "fecha_nacimiento"] = nr[0]["dob"]
    nutrition_df.at[indice, "sexo_nominal"] = nr[0]["sexo"]
    nutrition_df.at[indice, "sexo_archivo_origen"] = nominal_path.name
    nutrition_df.at[indice, "sexo_hoja_origen"] = nominal_sheet_name
    nutrition_df.at[indice, "sexo_fila_origen"] = nr[0]["fila"]
    nutrition_df.at[indice, "estado_vinculo_nominal"] = "unico_mes_y_nacimiento_coinciden"
    nutrition_df.at[indice, "observaciones_calidad"] = "; ".join(
        x for x in [r["observaciones_calidad"], "fecha de nacimiento de nutrición reemplazada por la de la ficha nominal"] if x)
fecha_nacimiento_corregida = pd.DataFrame(filas_nac)
print("Evaluaciones con la fecha de nacimiento de la ficha nominal:", len(fecha_nacimiento_corregida))
show_table(fecha_nacimiento_corregida.set_index("id_evaluacion"), "Fecha de nacimiento: nutrición → ficha nominal")
export_evidence("14_fecha_nacimiento_corregida_con_ficha_nominal.xlsx", {"evaluaciones": fecha_nacimiento_corregida})

Evaluaciones con la fecha de nacimiento de la ficha nominal: 10


,id_persona,periodo,fecha_nacimiento_nutricion,fecha_nacimiento_ficha_nominal,archivo_origen,hoja_origen,fila_origen
id_evaluacion,,,,,,,
NUT_202601_PPA_FN_0004,PPA_FN_0004,2026-01,2023-01-09,2023-09-01,Nutricion_IDs_202601 - ENERO - TODOS.xlsx,INFANTES,15
NUT_202601_PPA_FN_0028,PPA_FN_0028,2026-01,2015-01-30,2015-01-31,Nutricion_IDs_202601 - ENERO - TODOS.xlsx,MATOGROSO MUJERES,14
NUT_202601_PPA_FN_0066,PPA_FN_0066,2026-01,2016-09-02,2016-02-09,Nutricion_IDs_202601 - ENERO - TODOS.xlsx,MATOGROSO HOMBRES,7
NUT_202601_PPA_FN_0064,PPA_FN_0064,2026-01,2013-01-24,2013-01-28,Nutricion_IDs_202601 - ENERO - TODOS.xlsx,MATOGROSO HOMBRES,30
NUT_202604_PPA_FN_0004,PPA_FN_0004,2026-04,2023-01-09,2023-09-01,Nutricion_IDs_202604 - ABRIL - TODOS.xlsx,INFANTES (ABR),13
NUT_202604_PPA_FN_0028,PPA_FN_0028,2026-04,2015-01-30,2015-01-31,Nutricion_IDs_202604 - ABRIL - TODOS.xlsx,MATOGROSO MUJERES,14
NUT_202604_PPA_FN_0066,PPA_FN_0066,2026-04,2016-09-02,2016-02-09,Nutricion_IDs_202604 - ABRIL - TODOS.xlsx,MATOGROSO HOMBRES,7
NUT_202606_PPA_FN_0004,PPA_FN_0004,2026-06,2023-01-09,2023-09-01,Nutricion_IDs_202606 - JUNIO- TODOS.xlsx,INFANTES,13
NUT_202606_PPA_FN_0028,PPA_FN_0028,2026-06,2015-01-30,2015-01-31,Nutricion_IDs_202606 - JUNIO- TODOS.xlsx,MATOGROSO MUJERES,11


Evidencia guardada: C - KDD Steps/C - 3: Data Cleaning and Preprocessing/14_fecha_nacimiento_corregida_con_ficha_nominal.xlsx  (evaluaciones: 10 filas)


# 4. Exclusión de evaluaciones de nutrición

Antes de aplicar un tratamiento se guarda una copia intacta de lo extraído (`nutrition_df_bruto`), de modo que lo eliminado siempre se pueda recuperar.

In [ ]:
# Copia de seguridad de lo extraído, ANTES de aplicar cualquier tratamiento (nunca se modifican los datos originales)
nutrition_df_bruto = nutrition_df.copy()
print("Evaluaciones extraídas de los Excel:", len(nutrition_df_bruto))

# TRATAMIENTO 1 (lado nutrición): ELIMINAR LAS EVALUACIONES DE LOS CÓDIGOS INCONSISTENTES
mask_ids_excluidos = nutrition_df["id_persona"].isin(conflicting_ids)
evaluaciones_ids_excluidas = nutrition_df[mask_ids_excluidos].copy()
nutrition_df = nutrition_df[~mask_ids_excluidos].reset_index(drop=True)
print("Evaluaciones excluidas por código inconsistente:", len(evaluaciones_ids_excluidas))

# TRATAMIENTO 2: ELIMINAR LAS EVALUACIONES CON "PROVINCIA"
# Se interpreta como: ese mes, para esa persona, NO hubo evaluación nutricional.
mask_provincia = nutrition_df["estado_mediciones"] == "sin_medicion_provincia"
evaluaciones_provincia = nutrition_df[mask_provincia].copy()
nutrition_df = nutrition_df[~mask_provincia].reset_index(drop=True)
print("Evaluaciones excluidas por PROVINCIA:", len(evaluaciones_provincia))

# Para dejar constancia en el consolidado de por qué ese mes no tiene nutrición
provincia_keys = set(zip(evaluaciones_provincia["periodo"], evaluaciones_provincia["id_persona"]))

print("Evaluaciones que continúan al consolidado:", len(nutrition_df))
assert len(nutrition_df_bruto) == len(nutrition_df) + len(evaluaciones_ids_excluidas) + len(evaluaciones_provincia)

resumen_exclusiones = pd.DataFrame([
    ("Evaluaciones extraídas", len(nutrition_df_bruto)),
    ("Excluidas por código (ID) inconsistente", len(evaluaciones_ids_excluidas)),
    ("Excluidas por PROVINCIA", len(evaluaciones_provincia)),
    ("Evaluaciones que continúan", len(nutrition_df)),
], columns=["concepto", "evaluaciones"]).set_index("concepto")


show_table(resumen_exclusiones, "Efecto de las exclusiones sobre la nutrición")
show_table(pd.crosstab(evaluaciones_provincia["periodo"], evaluaciones_provincia["grupo_hogar"], margins=True, margins_name="TOTAL"),
           "Evaluaciones con PROVINCIA excluidas, por mes y hogar")

export_evidence("01_ids_inconsistentes_excluidos.xlsx", {
    "resumen_por_id": resumen_ids_excluidos.reset_index(),
    "fichas_nominales_excluidas": nominal_excluida,
    "evaluaciones_nutricion_excluidas": evaluaciones_ids_excluidas})
export_evidence("02_evaluaciones_provincia_excluidas.xlsx", {"evaluaciones_provincia": evaluaciones_provincia})

Evaluaciones extraídas de los Excel: 402
Evaluaciones excluidas por código inconsistente: 24
Evaluaciones excluidas por PROVINCIA: 23
Evaluaciones que continúan al consolidado: 355


,evaluaciones
concepto,
Evaluaciones extraídas,402
Excluidas por código (ID) inconsistente,24
Excluidas por PROVINCIA,23
Evaluaciones que continúan,355


grupo_hogar,MATOGROSO HOMBRES,MATOGROSO MUJERES,TOTAL
periodo,,,
2026-01,5,4,9
2026-04,4,3,7
2026-06,4,3,7
TOTAL,13,10,23


Evidencia guardada: C - KDD Steps/C - 3: Data Cleaning and Preprocessing/01_ids_inconsistentes_excluidos.xlsx  (resumen_por_id: 8 filas, fichas_nominales_excluidas: 169 filas, evaluaciones_nutricion_excluidas: 24 filas)
Evidencia guardada: C - KDD Steps/C - 3: Data Cleaning and Preprocessing/02_evaluaciones_provincia_excluidas.xlsx  (evaluaciones_provincia: 23 filas)


**Comentario 8:**

- **Tratamiento 1 — Códigos inconsistentes:** se excluyen 24 evaluaciones (3 por cada uno de los 8 códigos).
- **Tratamiento 2 — `PROVINCIA`:** se excluyen 23 evaluaciones de los hogares Matogroso (enero 9, abril 7, junio 7). No son «peso cero» ni datos perdidos: indican que ese mes esa persona no fue evaluada, y por eso no deben aparecer como evaluación ni contar en promedios o coberturas. La ficha nominal de esa persona y mes sí se conserva, con la marca `nutricion_excluida_motivo = PROVINCIA`, para dejar constancia de por qué no tiene nutrición.
- **Control:** 402 = 355 + 24 + 23 (se verifica con una comprobación automática).

Cada exclusión se guarda en su propio Excel: `01_ids_inconsistentes_excluidos.xlsx` (resumen, fichas excluidas y evaluaciones excluidas) y `02_evaluaciones_provincia_excluidas.xlsx`.

# 5. Normalización del texto de los diagnósticos

In [ ]:
# TRATAMIENTO 3: NORMALIZACIÓN DEL TEXTO DE LOS DIAGNÓSTICOS
# La normalización se aplicó durante la extracción (normalize_diagnosis): quita espacios y saltos de línea,
# pasa a MAYÚSCULAS y corrige los errores tipográficos conocidos.
# El texto tal como llegó sigue en las columnas *_original.

identificadores = ["id_evaluacion", "id_persona", "periodo", "archivo_origen", "hoja_origen", "fila_origen"]
columnas_dx_original = [c for c in nutrition_df.columns if c.startswith("dx_") and c.endswith("_original")]

dx_largo = nutrition_df[identificadores + columnas_dx_original].melt(
    id_vars=identificadores, var_name="columna_dx", value_name="valor_original")
dx_largo["columna_dx"] = dx_largo["columna_dx"].str.replace("_original", "", regex=False)
dx_largo = dx_largo[dx_largo["valor_original"].map(lambda v: isinstance(v, str) and clean_text(v) not in ("", "*"))].copy()
dx_largo["valor_normalizado"] = dx_largo["valor_original"].map(normalize_diagnosis)

def tipo_cambio(original, normalizado):
    visible = clean_text(original)
    if original == normalizado: return "sin_cambio"
    if visible == normalizado: return "solo_espacios_o_saltos_de_linea"
    if visible.upper() == normalizado: return "mayusculas_minusculas"
    return "variante_o_error_tipografico"

dx_largo["tipo_cambio"] = [tipo_cambio(o, n) for o, n in zip(dx_largo["valor_original"], dx_largo["valor_normalizado"])]
show_table(dx_largo["tipo_cambio"].value_counts().rename("celdas"), "Celdas de diagnóstico según el tratamiento de texto aplicado")

variantes_dx = (dx_largo[dx_largo["tipo_cambio"] != "sin_cambio"]
                .assign(valor_original_visible=lambda d: d["valor_original"].map(repr))
                .groupby(["valor_original_visible", "valor_normalizado", "tipo_cambio"]).size()
                .rename("frecuencia").reset_index().sort_values("frecuencia", ascending=False))

# Verificación: cada columna de diagnóstico debe quedar con una sola escritura por categoría
for columna in [c[:-9] for c in columnas_dx_original]:
    valores = nutrition_df[columna].dropna()
    assert (valores == valores.map(lambda v: normalize_diagnosis(v))).all(), columna
    assert not valores.str.contains(r"[a-z\n]|^\s|\s$|\s{2}", regex=True).any(), columna
print("Verificación superada: todos los diagnósticos quedaron en MAYÚSCULAS, sin espacios sobrantes ni variantes conocidas.")

categorias_finales = (dx_largo.assign(indicador=dx_largo["columna_dx"].str.extract(r"dx_(\w+?)_(?:curva|tabla)")[0],
                                      bloque=dx_largo["columna_dx"].str.extract(r"_(curva|tabla)$")[0])
                      .groupby(["indicador", "bloque", "valor_normalizado"]).size().rename("frecuencia").reset_index())
show_table(categorias_finales.set_index(["indicador", "bloque"]), "Categorías que quedan después de normalizar")

export_evidence("03_normalizacion_texto_aplicada.xlsx", {
    "resumen_por_tipo_de_cambio": dx_largo["tipo_cambio"].value_counts().rename("celdas").reset_index(),
    "variantes_y_normalizacion": variantes_dx,
    "categorias_resultantes": categorias_finales,
    "celdas_modificadas_detalle": dx_largo[dx_largo["tipo_cambio"] != "sin_cambio"]})

,celdas
tipo_cambio,
sin_cambio,941
solo_espacios_o_saltos_de_linea,554
mayusculas_minusculas,103
variante_o_error_tipografico,46


Verificación superada: todos los diagnósticos quedaron en MAYÚSCULAS, sin espacios sobrantes ni variantes conocidas.


Evidencia guardada: C - KDD Steps/C - 3: Data Cleaning and Preprocessing/03_normalizacion_texto_aplicada.xlsx  (resumen_por_tipo_de_cambio: 4 filas, variantes_y_normalizacion: 36 filas, categorias_resultantes: 45 filas, celdas_modificadas_detalle: 703 filas)


**Comentario 9:** Sobre las 355 evaluaciones que continúan hay 1 644 celdas con una categoría escrita, 941 ya estaban limpias y 703 (43 %) se normalizaron: 554 por espacios o saltos de línea sobrantes, 103 por mayúsculas/minúsculas y 46 por variantes o errores de escritura (incluye `ALTA` → `ALTO`).

Se verifica automáticamente que no quede ninguna variante, todas las columnas de diagnóstico terminan en mayúsculas, sin espacios sobrantes ni errores conocidos, de modo que cada categoría tiene una sola escritura.

El texto tal como llegó no se pierde, ya que está en las columnas `dx_…_original`. Se guardan en Excel las variantes y sus formas normalizadas (`03_normalizacion_texto_aplicada.xlsx`).

# 6. Completar el IMC faltante

In [ ]:
# TRATAMIENTO 4: COMPLETAR EL IMC FALTANTE
# IMC = peso (kg) / talla (m)^2. Ya existía la columna imc_calculado (peso y talla válidos). No se sobrescribe imc_reportado:
# se crea imc_completado, que usa el valor reportado y, solo si no existe, el calculado por nosotros.

sin_imc_reportado = nutrition_df["imc_reportado"].isna() & nutrition_df["imc_calculado"].notna()

nutrition_df["imc_completado"] = nutrition_df["imc_reportado"].where(~sin_imc_reportado, nutrition_df["imc_calculado"])
nutrition_df["imc_fue_calculado_por_nosotros"] = sin_imc_reportado

print("IMC completados por nosotros:", int(sin_imc_reportado.sum()))
print("Evaluaciones que siguen sin IMC (sin peso o sin talla):", int(nutrition_df["imc_completado"].isna().sum()))

imc_completados = nutrition_df.loc[sin_imc_reportado, ["id_evaluacion", "id_persona", "periodo", "peso_kg", "talla_cm",
                                                       "imc_reportado", "imc_original", "imc_calculado", "imc_completado",
                                                       "archivo_origen", "hoja_origen", "fila_origen"]]
show_table(imc_completados.set_index("id_evaluacion"), "IMC completados")
export_evidence("04_imc_completado.xlsx", {"imc_completados": imc_completados})

IMC completados por nosotros: 6
Evaluaciones que siguen sin IMC (sin peso o sin talla): 0


,id_persona,periodo,peso_kg,talla_cm,imc_reportado,imc_original,imc_calculado,imc_completado,archivo_origen,hoja_origen,fila_origen
id_evaluacion,,,,,,,,,,,
NUT_202603_PPA_FN_0117,PPA_FN_0117,2026-03,14.04,92.5,—,—,16.409058,16.409058,Nutricion_IDs_202603 - MARZO- INFANTES.xlsx,Eva.Nut marzo,5
NUT_202603_PPA_FN_0140,PPA_FN_0140,2026-03,14.30,85.0,—,—,19.792388,19.792388,Nutricion_IDs_202603 - MARZO- INFANTES.xlsx,Eva.Nut marzo,6
NUT_202603_PPA_FN_0157,PPA_FN_0157,2026-03,11.30,77.0,—,—,19.058863,19.058863,Nutricion_IDs_202603 - MARZO- INFANTES.xlsx,Eva.Nut marzo,7
NUT_202603_PPA_FN_0153,PPA_FN_0153,2026-03,10.40,73.5,—,—,19.251238,19.251238,Nutricion_IDs_202603 - MARZO- INFANTES.xlsx,Eva.Nut marzo,8
NUT_202603_PPA_FN_0146,PPA_FN_0146,2026-03,15.40,91.0,—,—,18.596788,18.596788,Nutricion_IDs_202603 - MARZO- INFANTES.xlsx,Eva.Nut marzo,9
NUT_202603_PPA_FN_0149,PPA_FN_0149,2026-03,10.50,73.5,—,—,19.436346,19.436346,Nutricion_IDs_202603 - MARZO- INFANTES.xlsx,Eva.Nut marzo,10


Evidencia guardada: C - KDD Steps/C - 3: Data Cleaning and Preprocessing/04_imc_completado.xlsx  (imc_completados: 6 filas)


**Comentario 10:** Las 6 evaluaciones de marzo (PPA_FN_0117, 0140, 0157, 0153, 0146 y 0149) tenían peso y talla pero no IMC, porque la plantilla de marzo reemplazó esa columna por el perímetro cefálico. Se calcularon con IMC = peso ÷ talla² (resultados entre 16.4 y 19.8) y quedan en la columna `imc_completado`, junto con el indicador `imc_fue_calculado_por_nosotros = VERDADERO`.

Cuando el IMC reportado difiere en más de 0.11 y el IMC se puede calcular, entonces se debería conservar el calculado en `imc_completado`. Los casos quedan en `imc_corregido_por_diferencia` y en `11_imc_corregido_por_diferencia.xlsx`.

# 7. Cálculo de los diagnósticos con asterisco usando las tablas del INS

Un `*` en un diagnóstico significa que no se calculó. Según el Paso 2, solo es un valor realmente faltante cuando el indicador **aplica por edad**: menores de 5 años usan P/E, T/E y P/T; de 5 años a más usan T/E e IMC. Se calcula con las **tablas de valoración nutricional antropométrica del INS/CENAN**, que son las que el hogar usa en su bloque «tabla de diagnóstico rápido».

In [ ]:
# TABLAS INS (CENAN) TRANSCRITAS A CSV: se cargan y se comprueba que estén completas y ordenadas
def leer_tabla(nombre):
    return pd.read_csv(ins_tables_dir / nombre)

TABLAS = {}
for sexo, etiqueta_muj_var, etiqueta_ninas_ninos in [("F", "mujeres", "ninas"), ("M", "varones", "ninos")]:
    TABLAS[("talla_5_19", sexo)] = leer_tabla(f"INS_5a19_talla_edad_{etiqueta_muj_var}.csv").set_index("edad_meses")
    TABLAS[("imc_5_19", sexo)] = leer_tabla(f"INS_5a19_imc_edad_{etiqueta_muj_var}.csv").set_index("edad_meses")
    TABLAS[("pe_te_menor5", sexo)] = leer_tabla(f"INS_menor5_peso_talla_edad_{etiqueta_ninas_ninos}.csv").set_index("edad_meses")
    for posicion in ("echado", "pie"):
        TABLAS[("pt_menor5_" + posicion, sexo)] = leer_tabla(
            f"INS_menor5_peso_talla_{etiqueta_ninas_ninos}_{posicion}.csv").set_index("talla_cm")

# Controles de integridad: tamaños esperados y valores que no decrecen de izquierda a derecha en cada fila
esperado = {"talla_5_19": 61, "imc_5_19": 61, "pe_te_menor5": 60, "pt_menor5_echado": 46, "pt_menor5_pie": 41}
control_tablas = []
for (nombre, sexo), tabla in TABLAS.items():
    valores = tabla.drop(columns=["sexo", "posicion"], errors="ignore")
    assert len(tabla) == esperado[nombre], (nombre, sexo, len(tabla))
    assert (valores.diff(axis=1).iloc[:, 1:] >= 0).all().all(), (nombre, sexo)
    control_tablas.append({"tabla": nombre, "sexo": sexo, "filas": len(tabla), "columnas_de_valores": valores.shape[1]})
show_table(pd.DataFrame(control_tablas).set_index("tabla"), "Tablas INS cargadas y verificadas")

,sexo,filas,columnas_de_valores
tabla,,,
talla_5_19,F,61,4
imc_5_19,F,61,4
pe_te_menor5,F,60,7
pt_menor5_echado,F,46,6
pt_menor5_pie,F,41,6
talla_5_19,M,61,4
imc_5_19,M,61,4
pe_te_menor5,M,60,7
pt_menor5_echado,M,46,6


**Comentario 11:** Los cuatro PDF del INS se transcribieron manualmente a 10 archivos CSV (por sexo): talla/edad e IMC/edad para 5 a 19 años (percentiles, CDC 2000), y peso/edad, talla/edad y peso/talla para menores de 5 años (desviaciones estándar, OMS 2006).


In [ ]:
# REGLAS DE CLASIFICACIÓN SEGÚN LAS TABLAS INS
#  <5 años (OMS 2006, desviaciones estándar):
#     P/E : < -2DE DESNUTRIDO | -2DE a +2DE NORMAL | > +2DE SOBREPESO
#     T/E : < -3DE TALLA BAJA SEVERA | -3DE a < -2DE TALLA BAJA | -2DE a +2DE NORMAL | > +2DE ALTO
#     P/T : < -3DE DESNUTRIDO SEVERO | -3DE a < -2DE DESNUTRIDO | -2DE a +2DE NORMAL | > +2DE a +3DE SOBREPESO | > +3DE OBESIDAD
#  5 a 19 años (CDC 2000, percentiles):
#     T/E : < P5 TALLA BAJA | P5 a P95 NORMAL | > P95 ALTO
#     IMC : < P5 DELGADEZ | P5 a < P95 NORMAL | >= P95 OBESIDAD   (el IMC se usa con un decimal, sin redondear)
# Indicadores que aplican por edad (regla de negocio): <5 años -> P/E, T/E, P/T ; >=5 años -> T/E, IMC.


APLICA_POR_EDAD = {"menor_5": {"pe", "te", "pt"}, "desde_5": {"te", "imc"}}
COLUMNA_INS = {"pe": "dx_pe_tabla", "te": "dx_te_tabla", "pt": "dx_pt_tabla", "imc": "dx_imc_tabla"}

def fila_5_19(meses):
    """Fila de la tabla 5-19 años: si los meses no coinciden se toma la edad anterior (pasos de 3 meses). Última fila: 19a11m."""
    if meses < 60 or meses >= 240:
        return None
    return 239 if meses >= 239 else (meses // 3) * 3

def clasificar_menor5(indicador, sexo, meses, peso, talla):
    """Devuelve (categoria, detalle) o (None, motivo_por_el_que_no_se_pudo)."""
    if indicador in ("pe", "te"):
        fila = TABLAS[("pe_te_menor5", sexo)].loc[meses]
        if indicador == "pe":
            if pd.isna(peso): return None, "falta_peso"
            cat = "DESNUTRIDO" if peso < fila["pe_m2de"] else "SOBREPESO" if peso > fila["pe_p2de"] else "NORMAL"
            return cat, f"peso {peso} kg vs -2DE={fila['pe_m2de']} / +2DE={fila['pe_p2de']} ({meses} meses)"
        if pd.isna(talla): return None, "falta_talla"
        if talla < fila["te_m3de"]: cat = "TALLA BAJA SEVERA"
        elif talla < fila["te_m2de"]: cat = "TALLA BAJA"
        elif talla <= fila["te_p2de"]: cat = "NORMAL"
        else: cat = "ALTO"
        return cat, f"talla {talla} cm vs -3DE={fila['te_m3de']} / -2DE={fila['te_m2de']} / +2DE={fila['te_p2de']} ({meses} meses)"
    # P/T: se busca por talla (cm entero hacia abajo) en la tabla "echado" (<2 años) o "de pie" (>=2 años)
    if pd.isna(peso) or pd.isna(talla): return None, "faltan_mediciones"
    posicion = "echado" if meses < 24 else "pie"
    tabla = TABLAS[("pt_menor5_" + posicion, sexo)]
    talla_fila = int(np.floor(talla))
    if talla_fila not in tabla.index: return None, f"talla_fuera_de_la_tabla_{posicion}"
    fila = tabla.loc[talla_fila]
    if peso < fila["pt_m3de"]: cat = "DESNUTRIDO SEVERO"
    elif peso < fila["pt_m2de"]: cat = "DESNUTRIDO"
    elif peso <= fila["pt_p2de"]: cat = "NORMAL"
    elif peso <= fila["pt_p3de"]: cat = "SOBREPESO"
    else: cat = "OBESIDAD"
    return cat, f"peso {peso} kg vs talla {talla_fila} cm ({posicion}): -3DE={fila['pt_m3de']} / -2DE={fila['pt_m2de']} / +2DE={fila['pt_p2de']} / +3DE={fila['pt_p3de']}"

def clasificar_desde5(indicador, sexo, meses, peso, talla):
    fila_edad = fila_5_19(meses)
    if fila_edad is None: return None, "edad_fuera_del_rango_5_a_19_anios"
    if indicador == "te":
        if pd.isna(talla): return None, "falta_talla"
        fila = TABLAS[("talla_5_19", sexo)].loc[fila_edad]
        cat = "TALLA BAJA" if talla < fila["p5"] else "ALTO" if talla > fila["p95"] else "NORMAL"
        return cat, f"talla {talla} cm vs P5={fila['p5']} / P95={fila['p95']} (fila {fila_edad} meses)"
    if pd.isna(peso) or pd.isna(talla): return None, "faltan_mediciones"
    imc = np.floor(peso / (talla / 100) ** 2 * 10 + 1e-9) / 10         # un decimal, sin redondear
    fila = TABLAS[("imc_5_19", sexo)].loc[fila_edad]
    cat = "DELGADEZ" if imc < fila["p5"] else "OBESIDAD" if imc >= fila["p95"] else "NORMAL"
    return cat, f"IMC {imc} vs P5={fila['p5']} / P95={fila['p95']} (fila {fila_edad} meses)"

def clasificar_ins(indicador, sexo, meses, peso, talla):
    grupo = "menor_5" if meses < 60 else "desde_5"
    if indicador not in APLICA_POR_EDAD[grupo]:
        return None, "no_aplica_por_edad"
    return (clasificar_menor5 if grupo == "menor_5" else clasificar_desde5)(indicador, sexo, meses, peso, talla)

# Pruebas rápidas con casos que se pueden verificar a mano contra las tablas impresas
assert clasificar_ins("pe", "F", 0, 3.0, 50)[0] == "NORMAL"                  # niña recién nacida 3.0 kg (límites 2.4 - 4.2)
assert clasificar_ins("pe", "M", 12, 7.0, 75)[0] == "DESNUTRIDO"             # niño de 1 año, 7.0 kg < 7.7 kg (-2DE)
assert clasificar_ins("te", "F", 24, 10.0, 75.5)[0] == "TALLA BAJA SEVERA"     # niña de 2 años, 75.5 cm < 76.0 cm (-3DE)
assert clasificar_ins("pt", "M", 30, 9.5, 85)[0] == "DESNUTRIDO"             # niño de pie, 85 cm: entre -3DE (9.2) y -2DE (10.0)
assert clasificar_ins("pt", "M", 30, 11.6, 85)[0] == "NORMAL"                # mismo niño con 11.6 kg: entre -1DE y +1DE
assert clasificar_ins("imc", "F", 120, 40.2, 132)[0] == "OBESIDAD"           # niña 10 años: IMC 23.0 >= P95 (22.9)
assert clasificar_ins("te", "M", 84, 20.0, 112.0)[0] == "TALLA BAJA"          # niño 7 años: 112 cm < P5 (113.1)
assert fila_5_19(239) == 239 and fila_5_19(238) == 237 and fila_5_19(61) == 60 and fila_5_19(240) is None
print("Pruebas de las reglas superadas.")

Pruebas de las reglas superadas.


**Comentario 12:** Las reglas salen literalmente de las instrucciones impresas en las tablas:

- **Menores de 5 años** (la fila se elige por la edad en meses; en peso/talla, por la talla en cm entera hacia abajo, en la tabla «acostado» si tiene menos de 2 años y «de pie» si tiene 2 o más):
  - **P/E:** menor que −2DE → `DESNUTRIDO`, entre −2DE y +2DE → `NORMAL`, mayor que +2DE → `SOBREPESO`.
  - **T/E:** menor que −3DE → `TALLA BAJA SEVERA`, de −3DE a menos de −2DE → `TALLA BAJA`, de −2DE a +2DE → `NORMAL`, mayor que +2DE → `ALTO`.
  - **P/T:** menor que −3DE → `DESNUTRIDO SEVERO`, de −3DE a menos de −2DE → `DESNUTRIDO`, de −2DE a +2DE → `NORMAL`, hasta +3DE → `SOBREPESO`, mayor que +3DE → `OBESIDAD`.
- **De 5 a 19 años** (si los meses no coinciden se toma la edad anterior, en pasos de 3 meses):
  - **T/E:** menor que P5 → `TALLA BAJA`, entre P5 y P95 → `NORMAL`, mayor que P95 → `ALTO`.
  - **IMC:** menor que P5 → `DELGADEZ`, de P5 a menos de P95 → `NORMAL`, P95 o más → `OBESIDAD`.

**Decisión de interpretación:** las tablas también definen «signos de alerta» (por ejemplo, talla entre −2DE y −1DE), pero no son una categoría de diagnóstico de la tabla, por lo que no se usan. Las pruebas automáticas de la celda verifican casos que se pueden comprobar a mano contra las tablas impresas.

In [ ]:
def parse_age_months(value):
    """Convierte la edad escrita a meses totales. Ej.: '7 años 3 meses' -> 87.
    Acepta las variantes observadas: año/años, mes/meses/mese, 'y', mayúsculas y espacios extra."""
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return None
    texto = clean_text(value).lower()
    anios = re.search(r"(\d+)\s*a[ñn]os?", texto)
    meses = re.search(r"(\d+)\s*mes", texto)
    if not anios and not meses:
        return None
    return (int(anios.group(1)) if anios else 0) * 12 + (int(meses.group(1)) if meses else 0)

# SEXO Y EDAD NECESARIOS PARA ELEGIR LA TABLA
# Sexo: se toma de la ficha nominal de cualquier mes (los códigos inconsistentes ya fueron excluidos, por lo que el sexo es único).
sexo_por_id = revision.groupby("id_persona")["sexo"].agg(lambda s: sorted(set(s)))
sexo_por_id = sexo_por_id[sexo_por_id.map(len) == 1].map(lambda s: s[0])

# Edad: se calcula como meses entre la fecha de nacimiento y el MES de la evaluación (el mes viene del nombre del libro).
# La edad escrita en el Excel se conserva solo como contraste: en el Paso 2 se vio que en junio viene adelantada 3-4 meses
# porque era una fórmula con TODAY recalculada en una fecha posterior a la evaluación.
nutrition_df["edad_texto_usada"] = nutrition_df["edad_reportada"].where(nutrition_df["edad_reportada"].notna(),
                                                                      nutrition_df["edad_formula_guardada"])
nutrition_df["edad_meses_texto_excel"] = nutrition_df["edad_texto_usada"].map(parse_age_months)

periodo_meses = nutrition_df["periodo"].map(lambda p: int(p[:4]) * 12 + int(p[5:7]))
nacimiento_meses = pd.to_datetime(nutrition_df["fecha_nacimiento"]).map(lambda d: d.year * 12 + d.month)
nutrition_df["edad_meses_usada_para_tablas_ins"] = periodo_meses - nacimiento_meses
nutrition_df["sexo_usado_para_tablas_ins"] = nutrition_df["id_persona"].map(sexo_por_id)

diferencia_edad = nutrition_df["edad_meses_texto_excel"] - nutrition_df["edad_meses_usada_para_tablas_ins"]
show_table(diferencia_edad.groupby(nutrition_df["periodo"]).agg(["count", "mean", "min", "max"]).round(2),
           "Edad escrita en el Excel menos edad calculada (meses), por mes")

print("Evaluaciones sin sexo disponible:", int(nutrition_df["sexo_usado_para_tablas_ins"].isna().sum()))
print("Evaluaciones sin edad calculable:", int(nutrition_df["edad_meses_usada_para_tablas_ins"].isna().sum()))
show_table(nutrition_df["sexo_usado_para_tablas_ins"].value_counts(dropna=False).rename("evaluaciones"), "Sexo usado para elegir la tabla")

,count,mean,min,max
periodo,,,,
2026-01,124,0.23,-12,12
2026-02,7,-1.43,-12,1
2026-03,6,0.33,0,1
2026-04,93,-0.52,-7,6
2026-06,93,3.89,-3,12
2026-08,32,-0.88,-1,0


Evaluaciones sin sexo disponible: 0
Evaluaciones sin edad calculable: 0


,evaluaciones
sexo_usado_para_tablas_ins,
M,204
F,151


**Comentario 13:**

- **Sexo:** se toma de la ficha nominal de cualquier mes, ya que el sexo no cambia y los códigos con sexo contradictorio ya fueron excluidos. Las 355 evaluaciones tienen sexo (204 varones y 151 mujeres), incluidas las de agosto, que no tienen ficha nominal propia.
- **Edad:** se calcula como los meses entre la fecha de nacimiento y el mes de la evaluación, y no con la edad escrita en el Excel. La tabla muestra por qué en junio la edad escrita está 3 a 4 meses adelantada (diferencia media de +3.3) por una fórmula con `TODAY` recalculada después de la evaluación, mientras que en los demás meses la diferencia es casi nula, con unos pocos errores de digitación puntuales. Usar la edad escrita habría clasificado a algunos niños con la fila o el indicador equivocado. La edad escrita se conserva en `edad_meses_texto_excel` solo como contraste.

In [ ]:
# TRATAMIENTO 5: CALCULAR LOS DIAGNÓSTICOS QUE VIENEN CON ASTERISCO (*) EN EL BLOQUE "TABLA DE DIAGNÓSTICO RÁPIDO INS"
# - Solo se calculan los '*' de indicadores que aplican por edad y que tienen mediciones y sexo disponibles.
# - No se sobrescribe nada original: dx_*_original sigue con el '*', y se agrega un flag TRUE/FALSE por cada columna de diagnóstico.
# - El bloque "percentiles OMS" no se calcula (el PDF de OMS solo trae gráficos, sin valores numéricos).

for indicador, columna in COLUMNA_INS.items():
    nutrition_df[columna + "_fue_calculado_por_nosotros"] = False

calculos, pendientes = [], []
for indice, r in nutrition_df.iterrows():
    for indicador, columna in COLUMNA_INS.items():
        if r[columna + "_estado"] != "asterisco_original":
            continue       # solo se tratan las celdas que vinieron con '*'
        base = {"id_evaluacion": r["id_evaluacion"], "id_persona": r["id_persona"], "periodo": r["periodo"],
                "indicador": indicador, "columna": columna, "sexo": r["sexo_usado_para_tablas_ins"],
                "edad_meses": r["edad_meses_usada_para_tablas_ins"], "peso_kg": r["peso_kg"], "talla_cm": r["talla_cm"],
                "archivo_origen": r["archivo_origen"], "hoja_origen": r["hoja_origen"], "fila_origen": r["fila_origen"]}
        if pd.isna(r["sexo_usado_para_tablas_ins"]) or pd.isna(r["edad_meses_usada_para_tablas_ins"]):
            pendientes.append({**base, "motivo": "sin_sexo_o_edad"}); continue
        categoria, detalle = clasificar_ins(indicador, r["sexo_usado_para_tablas_ins"],
                                            int(r["edad_meses_usada_para_tablas_ins"]), r["peso_kg"], r["talla_cm"])
        if categoria is None:
            pendientes.append({**base, "motivo": detalle})
        else:
            nutrition_df.at[indice, columna] = categoria
            nutrition_df.at[indice, columna + "_fue_calculado_por_nosotros"] = True
            calculos.append({**base, "categoria_calculada": categoria, "detalle_del_calculo": detalle})

calculos, pendientes = pd.DataFrame(calculos), pd.DataFrame(pendientes)
print("Diagnósticos calculados por nosotros:", len(calculos))
print("Asteriscos del bloque INS que no se calcularon:", len(pendientes))
show_table(calculos.set_index("id_evaluacion"), "Diagnósticos calculados con las tablas INS")
show_table(pendientes["motivo"].value_counts().rename("celdas"), "Asteriscos del bloque INS que se conservan, por motivo")

Diagnósticos calculados por nosotros: 3
Asteriscos del bloque INS que no se calcularon: 133


,id_persona,periodo,indicador,columna,sexo,edad_meses,peso_kg,talla_cm,archivo_origen,hoja_origen,fila_origen,categoria_calculada,detalle_del_calculo
id_evaluacion,,,,,,,,,,,,,
NUT_202601_PPA_FN_0057,PPA_FN_0057,2026-01,te,dx_te_tabla,F,216,59.8,155.0,Nutricion_IDs_202601 - ENERO - TODOS.xlsx,MIGUEL ECHENIQUE,24,NORMAL,talla 155.0 cm vs P5=152.4 / P95=173.7 (fila 216 meses)
NUT_202606_PPA_FN_0135,PPA_FN_0135,2026-06,pe,dx_pe_tabla,M,55,16.9,102.0,Nutricion_IDs_202606 - JUNIO- TODOS.xlsx,INFANTES,22,NORMAL,peso 16.9 kg vs -2DE=13.5 / +2DE=22.9 (55 meses)
NUT_202606_PPA_FN_0002,PPA_FN_0002,2026-06,pe,dx_pe_tabla,M,55,17.6,104.4,Nutricion_IDs_202606 - JUNIO- TODOS.xlsx,INFANTES,25,NORMAL,peso 17.6 kg vs -2DE=13.5 / +2DE=22.9 (55 meses)


,celdas
motivo,
no_aplica_por_edad,132
edad_fuera_del_rango_5_a_19_anios,1


**Comentario 14:** Entre las 355 evaluaciones hay 136 celdas con `*` en el bloque INS pero solo 3 corresponden a un valor faltante que se puede calcular**:

| Evaluación | Indicador | Dato | Resultado |
|---|---|---|---|
| PPA_FN_0057 (enero), mujer de 18 años | T/E | 155.0 cm, entre P5 (152.4) y P95 (173.7) | `NORMAL` |
| PPA_FN_0135 (junio), varón de 4 años y 7 meses | P/E | 16.9 kg, entre −2DE (13.5) y +2DE (22.9) | `NORMAL` |
| PPA_FN_0002 (junio), varón de 4 años y 7 meses | P/E | 17.6 kg, entre −2DE (13.5) y +2DE (22.9) | `NORMAL` |

Las otras 133 celdas se conservan con su `*`: 132 no aplican por edad (por ejemplo, P/E y P/T en mayores de 5 años, o IMC en menores de 5) y 1 corresponde a una persona de más de 19 años(PPA_FN_0054), fuera del rango de las tablas. En particular, PPA_FN_0121 (junio) tiene IMC con `*` y, con la edad calculada (4 años y 10 meses), el IMC efectivamente no aplica, con la edad escrita (5 años y 1 mes) habría parecido calculable, y se habría clasificado mal.

**Trazabilidad:** el valor original no se sobrescribe (`dx_…_original` mantiene el `*` y `dx_…_estado` mantiene «asterisco_original»). Cada diagnóstico de la tabla INS tiene su indicador propio, por ejemplo `dx_pe_tabla_fue_calculado_por_nosotros`(VERDADERO/FALSO). En `05_diagnosticos_calculados.xlsx` queda el detalle de cada cálculo (edad, sexo, peso, talla y límites usados) y en `06_asteriscos_que_se_conservan.xlsx` los que no se calcularon y por qué.

In [ ]:
# SENSIBILIDAD: ¿CAMBIARÍA EL RESULTADO SI LA EDAD SE TOMARA DEL TEXTO DEL EXCEL EN LUGAR DE CALCULARLA DESDE EL NACIMIENTO?
# Se recalculan las celdas anteriores con la edad escrita en el Excel y se compara la categoría.
comparacion = []
for _, c in calculos.iterrows():
    r = nutrition_df.loc[nutrition_df["id_evaluacion"] == c["id_evaluacion"]].iloc[0]
    meses_alt = int(r["edad_meses_texto_excel"])
    cat_alt, detalle_alt = clasificar_ins(c["indicador"], c["sexo"], meses_alt, c["peso_kg"], c["talla_cm"])
    comparacion.append({"id_evaluacion": c["id_evaluacion"], "indicador": c["indicador"],
                        "edad_meses_calculada": int(c["edad_meses"]), "categoria_con_edad_calculada": c["categoria_calculada"],
                        "edad_meses_texto_excel": meses_alt, "categoria_con_edad_del_excel": cat_alt or detalle_alt,
                        "coincide": cat_alt == c["categoria_calculada"]})
comparacion = pd.DataFrame(comparacion)
show_table(comparacion.set_index("id_evaluacion"), "Categoría con ambas fuentes de edad")
print("Casos en que cambia la categoría:", int((~comparacion["coincide"]).sum()))

export_evidence("05_diagnosticos_calculados.xlsx", {
    "diagnosticos_calculados": calculos,
    "sensibilidad_fuente_de_edad": comparacion})
export_evidence("06_asteriscos_que_se_conservan.xlsx", {
    "bloque_INS_no_calculados": pendientes,
    "resumen_por_motivo": pendientes["motivo"].value_counts().rename("celdas").reset_index() if len(pendientes) else pd.DataFrame()})

,indicador,edad_meses_calculada,categoria_con_edad_calculada,edad_meses_texto_excel,categoria_con_edad_del_excel,coincide
id_evaluacion,,,,,,
NUT_202601_PPA_FN_0057,te,216,NORMAL,216,NORMAL,True
NUT_202606_PPA_FN_0135,pe,55,NORMAL,59,NORMAL,True
NUT_202606_PPA_FN_0002,pe,55,NORMAL,58,NORMAL,True


Casos en que cambia la categoría: 0
Evidencia guardada: C - KDD Steps/C - 3: Data Cleaning and Preprocessing/05_diagnosticos_calculados.xlsx  (diagnosticos_calculados: 3 filas, sensibilidad_fuente_de_edad: 3 filas)
Evidencia guardada: C - KDD Steps/C - 3: Data Cleaning and Preprocessing/06_asteriscos_que_se_conservan.xlsx  (bloque_INS_no_calculados: 133 filas, resumen_por_motivo: 2 filas)


**Comentario 15:** Si se hubiera usado la edad escrita en el Excel en lugar de la calculada, las 3 categorías serían iguales (`NORMAL`), de modo que los diagnósticos calculados son robustos. La diferencia solo cambia qué celdas se consideran calculables (como el caso de PPA_FN_0121 explicado arriba). La comparación se guarda en la hoja `sensibilidad_fuente_de_edad` de `05_diagnosticos_calculados.xlsx`.

In [ ]:
# ASTERISCOS QUE QUEDAN EN TODO EL CONJUNTO (ambos bloques), CLASIFICADOS POR MOTIVO
# Permite ver, al final, cuántos '*' se resolvieron y cuántos se conservan y por qué.
filas = []
for indicador in ("pe", "te", "pt", "imc"):
    for bloque in ("curva", "tabla"):
        columna = f"dx_{indicador}_{bloque}"
        restantes = nutrition_df[nutrition_df[columna + "_estado"] == "asterisco_original"]
        for _, r in restantes.iterrows():
            meses = r["edad_meses_usada_para_tablas_ins"]
            if bloque == "tabla" and r[columna + "_fue_calculado_por_nosotros"]:
                motivo = "resuelto_calculado_por_nosotros"
            elif pd.isna(meses):
                motivo = "sin_edad"
            elif indicador not in APLICA_POR_EDAD["menor_5" if meses < 60 else "desde_5"]:
                motivo = "no_aplica_por_edad"
            elif meses >= 240:
                motivo = "edad_fuera_del_rango_5_a_19_anios"
            elif bloque == "curva":
                motivo = "bloque_OMS_sin_tabla_numerica"
            else:
                motivo = "bloque_INS_no_calculable (ver 06_asteriscos_que_se_conservan.xlsx)"
            filas.append({"indicador": indicador, "bloque": bloque, "motivo": motivo})
resumen_final_ast = pd.crosstab([pd.Series([f["indicador"] for f in filas], name="indicador"),
                                 pd.Series([f["bloque"] for f in filas], name="bloque")],
                                pd.Series([f["motivo"] for f in filas], name="motivo"), margins=True, margins_name="TOTAL")
show_table(resumen_final_ast, "Estado final de los asteriscos")
export_evidence("07_estado_final_asteriscos.xlsx", {"resumen": resumen_final_ast.reset_index()})

Evidencia guardada: C - KDD Steps/C - 3: Data Cleaning and Preprocessing/07_estado_final_asteriscos.xlsx  (resumen: 9 filas)


**Comentario 16:** de las evaluaciones que continúan: 3 se resolvieron (calculados por nosotros), 264 no aplican por edad (el `*` es correcto), 11 son del bloque «percentiles OMS» (P/T en niños de 2 a 4 años) y 5 pertenecen a una persona mayor de 19 años (PPA_FN_0054), fuera del rango de las tablas. Los del bloque OMS no se pueden calcular porque el PDF de OMS no trae valores numéricos.

# 7A. IMC que difiere en más de 0.11: se conserva el calculado

Cuando el IMC reportado difiere del calculado (peso ÷ talla², sin redondear) en más de 0.11 **y** el IMC se pudo calcular, se conserva el calculado. Se deja evidencia de cada NNA modificado con la **fila completa** de la nutrición consolidada, tal como estaba y como quedó.

In [ ]:
# TRATAMIENTO 6: IMC QUE DIFIERE EN MÁS DE 0.11 DEL CALCULADO -> SE CONSERVA EL CALCULADO
# Documentación del cálculo: IMC = peso (kg) / talla (m)^2, con todos los decimales y sin redondear (la misma fórmula de imc_calculado).
# Solo se corrige cuando el IMC se pudo calcular (hay peso y talla válidos, talla > 0) y el reportado difiere del calculado en más
# de IMC_REVIEW_TOLERANCE, en cualquier sentido. imc_reportado e imc_original NO se sobrescriben: el valor que se conserva queda en
# imc_completado, con un indicador TRUE/FALSE. Los diagnósticos (dx_…) no se recalculan: son los que registró el hogar.
nutrition_columns = nutrition_columns + [c for c in ["imc_corregido_por_diferencia"] if c not in nutrition_columns]

corregir_imc = (nutrition_df["imc_reportado"].notna() & nutrition_df["imc_calculado"].notna()
                & (nutrition_df["diferencia_imc"].abs() > IMC_REVIEW_TOLERANCE))

# La fila completa de la nutrición consolidada, TAL COMO ESTABA, antes de cambiar nada
imc_fila_antes = nutrition_df.loc[corregir_imc].copy()
imc_fila_antes["imc_corregido_por_diferencia"] = False

nutrition_df["imc_corregido_por_diferencia"] = corregir_imc
nutrition_df.loc[corregir_imc, "imc_completado"] = nutrition_df.loc[corregir_imc, "imc_calculado"]
imc_fila_despues = nutrition_df.loc[corregir_imc].copy()

print("Evaluaciones con IMC calculable y diferencia mayor a", IMC_REVIEW_TOLERANCE, ":", int(corregir_imc.sum()))
print("Evaluaciones que siguen sin IMC:", int(nutrition_df["imc_completado"].isna().sum()))

resumen_imc_corregido = pd.DataFrame({
    "id_evaluacion": imc_fila_antes["id_evaluacion"], "id_persona": imc_fila_antes["id_persona"],
    "periodo": imc_fila_antes["periodo"], "grupo_hogar": imc_fila_antes["grupo_hogar"], "casa": imc_fila_antes["casa"],
    "sexo": imc_fila_antes["sexo_usado_para_tablas_ins"], "edad_meses": imc_fila_antes["edad_meses_usada_para_tablas_ins"],
    "peso_kg": imc_fila_antes["peso_kg"], "talla_cm": imc_fila_antes["talla_cm"],
    "imc_reportado": imc_fila_antes["imc_reportado"],
    "imc_conservado_antes": imc_fila_antes["imc_completado"],
    "imc_conservado_ahora": imc_fila_despues["imc_completado"],
    "cambio": imc_fila_despues["imc_completado"] - imc_fila_antes["imc_completado"],
    "archivo_origen": imc_fila_antes["archivo_origen"], "hoja_origen": imc_fila_antes["hoja_origen"],
    "fila_origen": imc_fila_antes["fila_origen"]})
show_table(resumen_imc_corregido.set_index("id_evaluacion"), "NNA con IMC corregido (se conserva el calculado)")

# Controles: lo corregido es exactamente el calculado, y lo demás no se tocó
assert (nutrition_df.loc[corregir_imc, "imc_completado"] == nutrition_df.loc[corregir_imc, "imc_calculado"]).all()
assert nutrition_df.loc[~corregir_imc, "imc_completado"].equals(
    nutrition_df.loc[~corregir_imc, "imc_reportado"].where(~sin_imc_reportado[~corregir_imc], nutrition_df.loc[~corregir_imc, "imc_calculado"]))
assert nutrition_df["imc_completado"].notna().all()

export_evidence("11_imc_corregido_por_diferencia.xlsx", {
    "resumen_por_nna": resumen_imc_corregido,
    "fila_completa_antes": imc_fila_antes,
    "fila_completa_despues": imc_fila_despues})

Evaluaciones con IMC calculable y diferencia mayor a 0.11 : 9
Evaluaciones que siguen sin IMC: 0


,id_persona,periodo,grupo_hogar,casa,sexo,edad_meses,peso_kg,talla_cm,imc_reportado,imc_conservado_antes,imc_conservado_ahora,cambio,archivo_origen,hoja_origen,fila_origen
id_evaluacion,,,,,,,,,,,,,,,
NUT_202608_PPA_FN_0027,PPA_FN_0027,2026-08,MATOGROSO MUJERES,CASA ADOLESCENTES MUJERES PROVINCIA,F,191,58.30,156.4,24.20,24.20,23.833897,-0.366103,Nutricion_IDs_202608 - AGOSTO- OMG.xlsx,MATOGROSO MUJERES,5
NUT_202608_PPA_FN_0032,PPA_FN_0032,2026-08,MATOGROSO MUJERES,CASA ADOLESCENTES MUJERES PROVINCIA,F,166,45.00,146.6,21.40,21.40,20.938452,-0.461548,Nutricion_IDs_202608 - AGOSTO- OMG.xlsx,MATOGROSO MUJERES,6
NUT_202608_PPA_FN_0045,PPA_FN_0045,2026-08,MATOGROSO MUJERES,CASA ADOLESCENTES MUJERES PROVINCIA,F,206,57.70,159.1,23.08,23.08,22.794783,-0.285217,Nutricion_IDs_202608 - AGOSTO- OMG.xlsx,MATOGROSO MUJERES,9
NUT_202608_PPA_FN_0044,PPA_FN_0044,2026-08,MATOGROSO MUJERES,CASA ADOLESCENTES MUJERES PROVINCIA,F,210,75.40,153.5,31.80,31.80,32.000340,0.200340,Nutricion_IDs_202608 - AGOSTO- OMG.xlsx,MATOGROSO MUJERES,19
NUT_202608_PPA_FN_0067,PPA_FN_0067,2026-08,MATOGROSO HOMBRES,CASA ADOLESCENTES VARONES PROVINCIA,M,172,64.80,164.6,23.80,23.80,23.917476,0.117476,Nutricion_IDs_202608 - AGOSTO- OMG.xlsx,MATOGROSO HOMBRES,6
NUT_202608_PPA_FN_0080,PPA_FN_0080,2026-08,MATOGROSO HOMBRES,CASA ADOLESCENTES VARONES PROVINCIA,M,184,61.65,160.9,24.08,24.08,23.813377,-0.266623,Nutricion_IDs_202608 - AGOSTO- OMG.xlsx,MATOGROSO HOMBRES,14
NUT_202608_PPA_FN_0095,PPA_FN_0095,2026-08,MATOGROSO HOMBRES,CASA ADOLESCENTES VARONES PROVINCIA,M,169,61.65,160.9,24.08,24.08,23.813377,-0.266623,Nutricion_IDs_202608 - AGOSTO- OMG.xlsx,MATOGROSO HOMBRES,18
NUT_202608_PPA_FN_0099,PPA_FN_0099,2026-08,MATOGROSO HOMBRES,CASA ADOLESCENTES VARONES PROVINCIA,M,193,62.30,164.8,23.16,23.16,22.938955,-0.221045,Nutricion_IDs_202608 - AGOSTO- OMG.xlsx,MATOGROSO HOMBRES,20
NUT_202608_PPA_FN_0072,PPA_FN_0072,2026-08,MATOGROSO HOMBRES,CASA ADOLESCENTES VARONES PROVINCIA,M,218,59.00,161.5,22.76,22.76,22.620748,-0.139252,Nutricion_IDs_202608 - AGOSTO- OMG.xlsx,MATOGROSO HOMBRES,21


Evidencia guardada: C - KDD Steps/C - 3: Data Cleaning and Preprocessing/11_imc_corregido_por_diferencia.xlsx  (resumen_por_nna: 9 filas, fila_completa_antes: 9 filas, fila_completa_despues: 9 filas)


**Comentario 16A:** Las 9 evaluaciones son de agosto (hogares Matogroso): `PPA_FN_0027`, `0032`, `0045`, `0044`, `0067`, `0080`, `0095`, `0099` y `0072`. En 7 el IMC reportado era mayor que el calculado y en 2 menor; los cambios van de −0.46 a +0.20. En `imc_completado` queda el calculado (por ejemplo, `PPA_FN_0027`: 24.20 → 23.83), con `imc_corregido_por_diferencia = VERDADERO`.

**Evidencia:** `11_imc_corregido_por_diferencia.xlsx` con tres hojas: `resumen_por_nna`, `fila_completa_antes` y `fila_completa_despues` (las 85 columnas de la nutrición, una fila por NNA).

# 7B. Limpieza y homologación de la ficha nominal

Hasta aquí la ficha nominal se usó solo para validar identidades y se llevaba al consolidado tal como llegó: con comillas (`"M"`, `"2014-02-23"`), fechas en tres formatos y `NA` / `No tiene` dentro de columnas de fecha. Aquí se construye `df_nominal_limpio` sin tocar `df_nominal`

In [ ]:
# TRATAMIENTO 7: LIMPIEZA DE FORMATO Y HOMOLOGACIÓN DE LA FICHA NOMINAL (antes de la unión)
# Las celdas de la ficha nominal llegan tal como están en el Excel: con comillas ("M", "2014-02-23"), con fechas en tres formatos
# y con 'NA' / 'No tiene' dentro de columnas de fecha. Aquí se arma df_nominal_limpio. df_nominal NO se modifica: se sigue usando
# para validar identidades y para conservar el valor original en el consolidado.
# Reglas:
#  - Todas las celdas de texto: se quitan las comillas de los bordes, los espacios sobrantes y los saltos de línea.
#  - periodo: "2025_01" -> "2025-01" (misma forma que usa el resto del notebook). Código y sexo: solo se quitan las comillas.
#  - Fechas (nacimiento, ingreso, resolución, oficio y elaboración del PTI): se convierten a fecha real. Formatos observados:
#    AAAA-MM-DD, DD/MM/AAAA y DD/MM/AA (año de dos dígitos -> 20AA). 'NA' y 'No tiene' no son fechas: quedan vacías.
#  - Texto: las variantes de un mismo valor (mayúsculas, tildes, 'è' por 'é') se llevan a la escritura MÁS FRECUENTE de cada columna
#    (calculada con las fichas que continúan). Contenidos distintos (PERUANA/PERUANO) NO se unifican: requiere decisión de la entidad.
#  - 'VIGENCIA DEL PTI' mezcla fechas (ya en AAAA-MM-DD) con texto ('12 meses', 'Trimestral'): se trata como texto.

GRAVE_A_AGUDA = str.maketrans("àèìòùÀÈÌÒÙ", "áéíóúÁÉÍÓÚ")
POS_PERIODO, POS_ID, POS_SEXO = 0, 1, 2
POS_FECHAS = [3, 8, 12, 14, 17]                               # nacimiento, ingreso, resolución, oficio, elaboración del PTI
FUERA_DE_HOMOLOGACION = [POS_PERIODO, POS_ID, POS_SEXO] + POS_FECHAS
SIN_FECHA = {"na", "no tiene", ""}                            # valores que no son fecha (se comparan en minúsculas)
FORMATOS_FECHA = [("%Y-%m-%d", "AAAA-MM-DD"), ("%d/%m/%Y", "DD/MM/AAAA"), ("%d/%m/%y", "DD/MM/AA")]

def texto_visible(valor):
    """Texto sin comillas, con espacios y saltos de línea uniformes y tildes graves corregidas."""
    return clean_text(valor).translate(GRAVE_A_AGUDA)

def forma_canonica(valor):
    """Forma comparable: sin comillas, sin tildes y en minúsculas (agrupa las variantes de un mismo valor)."""
    texto = unicodedata.normalize("NFKD", clean_text(valor))
    return "".join(ch for ch in texto if not unicodedata.combining(ch)).casefold()

def escritura_mas_frecuente(grupo):
    return grupo.value_counts().sort_index().idxmax()          # en empate, la primera en orden alfabético

def fecha_limpia(valor):
    """Devuelve (fecha real o NaT, formato en que venía escrita)."""
    texto = clean_text(valor)
    if texto.casefold() in SIN_FECHA:
        return pd.NaT, "vacío (" + (texto or "sin valor") + ")"
    for formato, nombre in FORMATOS_FECHA:
        try:
            return pd.Timestamp(datetime.strptime(texto, formato)), nombre
        except ValueError:
            continue
    return pd.NaT, "NO RECONOCIDO"

df_nominal_bruto = df_nominal.copy()                            # copia intacta, tal como llegó
df_nominal_limpio = df_nominal.copy()
continua = ~mask_nominal_excluida                               # fichas que sí forman parte del consolidado

resumen_limpieza, mapa_cambios_texto, formatos_fecha, fechas_dos_digitos, muestra = [], [], [], [], []
for posicion, columna in enumerate(df_nominal_bruto.columns):
    serie = df_nominal_bruto[columna]
    con_comillas = int(serie.map(lambda v: isinstance(v, str) and '"' in v).sum())
    homologadas = 0
    if posicion == POS_PERIODO:
        limpio, tipo = serie.map(lambda v: clean_text(v).replace("_", "-")), "periodo: sin comillas, AAAA-MM"
    elif posicion in (POS_ID, POS_SEXO):
        limpio, tipo = serie.map(clean_text), "sin comillas"
    elif posicion in POS_FECHAS:
        resultado = serie.map(fecha_limpia)
        limpio = pd.to_datetime(resultado.map(lambda r: r[0]))
        formato = resultado.map(lambda r: r[1])
        formatos_fecha.append(formato[continua].value_counts().rename("filas").rename_axis("formato_original").reset_index().assign(columna=columna))
        if (formato == "DD/MM/AA").any():
            fechas_dos_digitos.append(pd.DataFrame({"fila_excel_origen": serie.index[formato == "DD/MM/AA"] + 2,
                                                    "id_persona": df_nominal_bruto.iloc[:, POS_ID][formato == "DD/MM/AA"].map(clean_text),
                                                    "columna": columna, "valor_original": serie[formato == "DD/MM/AA"].map(repr),
                                                    "fecha_limpia": limpio[formato == "DD/MM/AA"]}))
        tipo = "fecha real (AAAA-MM-DD)"
    elif not serie.map(lambda v: isinstance(v, str)).all():
        limpio, tipo = serie, "sin cambios (no es texto)"
    else:
        visible, canonico = serie.map(texto_visible), serie.map(forma_canonica)
        elegido_continua = visible[continua].groupby(canonico[continua]).agg(escritura_mas_frecuente)
        elegido_todas = visible.groupby(canonico).agg(escritura_mas_frecuente)
        limpio = canonico.map(lambda forma: elegido_continua.get(forma, elegido_todas[forma]))
        cambia = serie.str.strip('"') != limpio                 # cambio real, sin contar las comillas
        tipo = "texto homologado" if cambia.any() else "texto: solo comillas, espacios y saltos de línea"
        homologadas = int((cambia & continua).sum())
        if cambia.any():
            cambios = pd.DataFrame({"valor_original": serie[cambia & continua].map(repr), "valor_homologado": limpio[cambia & continua]})
            mapa_cambios_texto.append(cambios.groupby(["valor_original", "valor_homologado"]).size().rename("filas").reset_index().assign(columna=columna))
    df_nominal_limpio[columna] = limpio

    resumen_limpieza.append({"posicion": posicion, "columna": columna, "tipo_de_limpieza": tipo, "celdas_con_comillas": con_comillas,
                             "celdas_homologadas": homologadas,
                             "valores_distintos_antes": serie[continua].map(clean_text).nunique(),
                             "valores_distintos_despues": limpio[continua].nunique(),
                             "celdas_vacias_despues": int(limpio[continua].isna().sum())})
    pares = pd.DataFrame({"columna": columna, "original": serie[continua].map(repr),
                          "limpio": limpio[continua].map(lambda v: "(vacío)" if pd.isna(v) else v.strftime("%Y-%m-%d")
                                                         if isinstance(v, pd.Timestamp) else str(v))})
    muestra.append(pares.groupby(["columna", "original", "limpio"]).size().rename("filas").reset_index()
                   .sort_values("filas", ascending=False).head(3))

resumen_limpieza = pd.DataFrame(resumen_limpieza)
formatos_fecha = pd.concat(formatos_fecha)[["columna", "formato_original", "filas"]]
mapa_cambios_texto = pd.concat(mapa_cambios_texto)[["columna", "valor_original", "valor_homologado", "filas"]].sort_values(
    ["columna", "filas"], ascending=[True, False])
fechas_dos_digitos = pd.concat(fechas_dos_digitos)
muestra_antes_despues = pd.concat(muestra)

# Controles de la limpieza
assert not (formatos_fecha["formato_original"] == "NO RECONOCIDO").any(), "Hay fechas con un formato no reconocido."
for columna in df_nominal_limpio.columns[POS_FECHAS]:
    assert pd.api.types.is_datetime64_any_dtype(df_nominal_limpio[columna]), columna
for columna in df_nominal_limpio.columns:
    if df_nominal_limpio[columna].dtype == "object":
        assert not df_nominal_limpio[columna].astype(str).str.contains('"').any(), "Quedaron comillas en " + columna
# El nacimiento limpio es el mismo que se usó para validar las identidades
assert (df_nominal_limpio.iloc[:, 3].dt.strftime("%Y-%m-%d") == df_nominal_bruto.iloc[:, 3].map(parse_birth_date)).all()
# Una sola escritura por valor en cada columna homologada
for columna in resumen_limpieza.loc[resumen_limpieza["tipo_de_limpieza"] == "texto homologado", "columna"]:
    assert df_nominal_limpio[columna].groupby(df_nominal_bruto[columna].map(forma_canonica)).nunique().max() == 1, columna
# Los datos originales no se tocaron
assert df_nominal.equals(df_nominal_bruto)

print("Columnas de la ficha nominal:", len(df_nominal_limpio.columns), "| con fechas convertidas:", len(POS_FECHAS),
      "| con texto homologado:", int((resumen_limpieza["tipo_de_limpieza"] == "texto homologado").sum()))

show_table(resumen_limpieza.set_index("posicion"), "Efecto de la limpieza por columna (fichas que continúan)")


Columnas de la ficha nominal: 33 | con fechas convertidas: 5 | con texto homologado: 11


,columna,tipo_de_limpieza,celdas_con_comillas,celdas_homologadas,valores_distintos_antes,valores_distintos_despues,celdas_vacias_despues
posicion,,,,,,,
0,periodo,"periodo: sin comillas, AAAA-MM",2667,0,19,19,0
1,IDs_FichaNominal,sin comillas,0,0,149,149,0
2,Sexo,sin comillas,2667,0,2,2,0
3,Fecha de nacimiento,fecha real (AAAA-MM-DD),2667,0,142,142,0
4,Departamento,"texto: solo comillas, espacios y saltos de línea",2667,0,8,8,0
5,Provincia,"texto: solo comillas, espacios y saltos de línea",2667,0,10,10,0
6,Distrito,"texto: solo comillas, espacios y saltos de línea",2667,0,30,30,0
7,Nacionalidad,"texto: solo comillas, espacios y saltos de línea",2667,0,2,2,0
8,Fecha de ingreso,fecha real (AAAA-MM-DD),2667,0,99,99,0


In [ ]:
show_table(formatos_fecha.set_index("columna"), "Formatos de fecha encontrados (fichas que continúan)")

,formato_original,filas
columna,,
Fecha de nacimiento,AAAA-MM-DD,2391
Fecha de nacimiento,DD/MM/AAAA,107
Fecha de ingreso,AAAA-MM-DD,1644
Fecha de ingreso,DD/MM/AAAA,854
Fecha de la Resolucion Administrativa,AAAA-MM-DD,1931
Fecha de la Resolucion Administrativa,DD/MM/AAAA,366
Fecha de la Resolucion Administrativa,vacío (NA),182
Fecha de la Resolucion Administrativa,DD/MM/AA,19
Fecha del Oficio de Ingreso,AAAA-MM-DD,2052


In [ ]:
show_table(fechas_dos_digitos.drop_duplicates(["id_persona", "columna", "valor_original"]).set_index("id_persona"),
           "Fechas con año de dos dígitos (DD/MM/AA)")


,fila_excel_origen,columna,valor_original,fecha_limpia
id_persona,,,,
PPA_FN_0060,62,Fecha de la Resolucion Administrativa,"'""21/01/21""'",2021-01-21


In [ ]:
show_table(mapa_cambios_texto.set_index("columna"), "Homologación de texto (valor original → valor homologado)")

export_evidence("10_limpieza_ficha_nominal.xlsx", {
    "resumen_por_columna": resumen_limpieza,
    "formatos_de_fecha": formatos_fecha,
    "fechas_dos_digitos": fechas_dos_digitos,
    "mapa_de_cambios_texto": mapa_cambios_texto,
    "muestra_antes_despues": muestra_antes_despues})

,valor_original,valor_homologado,filas
columna,,,
ESTADO DEL PTI,"'""no tiene""'",No tiene,33
Estado de Salud Física Actual,"'""Saludable\n\n""'",Saludable,58
Estado de Salud Física Actual,"'""En proceso de evaluaciòn""'",En proceso de evaluación,4
Estado de Salud Física Actual,"'""Anemia\n\n""'",ANEMIA,3
Grado de Instrucción,"'""3ero de Secundaria""'",3ero de Secundaria,42
Grado de Instrucción,"'""5to de secundaria""'",5to de Secundaria,28
Grado de Instrucción,"'""3er Grado de Primaria""'",3er Grado de primaria,26
Grado de Instrucción,"'""Tècnico Productivo""'",Tecnico productivo,11
Grado de Instrucción,"'""4to de primaria""'",4to de Primaria,10


Evidencia guardada: C - KDD Steps/C - 3: Data Cleaning and Preprocessing/10_limpieza_ficha_nominal.xlsx  (resumen_por_columna: 33 filas, formatos_de_fecha: 13 filas, fechas_dos_digitos: 19 filas, mapa_de_cambios_texto: 46 filas, muestra_antes_despues: 91 filas)


**Comentario 16B:**

- **Comillas:** 31 de las 33 columnas traían todas sus celdas entre comillas (solo el código y `Comunicado_DGA` no). Se quitan junto con los espacios dobles y los saltos de línea.
- **Fechas reales (5 columnas):** nacimiento, ingreso, resolución administrativa, oficio de ingreso y elaboración del PTI. Venían como `AAAA-MM-DD` y `DD/MM/AAAA`, y 19 celdas de resolución como `DD/MM/AA` (`21/01/21` de `PPA_FN_0060` → 2021-01-21, coherente con su ingreso y oficio de enero de 2021). Las fechas con barras se leen como día/mes/año, igual que en la validación de identidades, el cuaderno comprueba que el nacimiento limpio coincide con el usado allí.
- **`NA` y `No tiene` en columnas de fecha:** no son fechas, quedan vacías(resolución 182, oficio 58 y PTI 1 017 fichas). Es una ausencia declarada, no un dato perdido.
- **Texto homologado (11 columnas, 1 420 celdas): las variantes de un mismo valor (`Si`/`si`, `No tiene`/`no tiene`, `Tècnico`/`Técnico`, `Resolución Admi.` con saltos de línea…) se llevan a la escritura más frecuente de cada columna, igual que en el Tratamiento 6 del otro cuaderno. Las otras 13 columnas de texto solo tenían comillas.
- **Periodo:** `2025_01` → `2025-01`.

**Qué no se unificó (lo decide la entidad):** diferencias de contenido como `PERUANA`/`PERUANO`, `UPE LIMA SUR`/`UPE-LIMA-SUR` o `No aplica`/`No aplicca`. `VIGENCIA DEL PTI` mezcla fechas con texto (`12 meses`, `Trimestral`), por lo que se trata como texto.

**Evidencia:** `10_limpieza_ficha_nominal.xlsx` (resumen por columna, formatos de fecha, fechas de dos dígitos, mapa de cambios de texto y muestra antes/después). El cuaderno verifica que no quede ninguna comilla, que las 5 columnas sean fechas reales, que cada valor tenga una sola escritura y que `df_nominal` siga igual que al llegar.

# 7C. Evaluación de los datos vacíos para el Paso 4

El Paso 4 construirá las variables que explican el historial y dejará **una fila por NNA**. Antes de unir las fuentes se revisa, columna por columna de la nutrición (ya tratada) y de la ficha nominal (ya limpia), cuánto está vacío, en cuántos meses hay dato y si el vacío es estructural o un dato que falta. Con eso se decide qué se conserva, qué se completa y qué no pasa al consolidado.

In [ ]:
# EVALUACIÓN DE LOS DATOS VACÍOS PARA EL PASO 4 (una fila por NNA con variables que explican el historial)
# Para cada columna de nutrición (ya tratada) y de la ficha nominal (ya limpia) se mide cuánto está vacío, en cuántos meses hay dato y
# si el vacío es "estructural" (el indicador no aplica por edad) o un dato que falta. Con eso se decide:
#   CONSERVAR  |  CONSERVAR (vacío con significado) |  COMPLETAR (se rellena con otra fuente) |  ELIMINAR
# "ELIMINAR en consolidado 2" quita la columna solo del consolidado limpio: el consolidado 1 la sigue teniendo como evidencia.
# Los numéricos (peso, talla, IMC) no tienen vacíos, los diagnósticos son categorías, por lo que un promedio de meses anteriores no aplica.

def es_vacio(serie):
    return serie.isna() | serie.map(lambda v: isinstance(v, str) and v.strip() == "")

nut = nutrition_df.drop(columns=["edad_texto_usada"])           # edad_texto_usada es solo un paso intermedio
meses_totales = nut["periodo"].nunique()
edad_nut = nut["edad_meses_usada_para_tablas_ins"]
aplica_dx = {"pe": edad_nut < 60, "pt": edad_nut < 60, "te": edad_nut.notna(), "imc": edad_nut >= 60}   # regla de negocio del tratamiento 5

# --- Decisiones sobre las columnas de nutrición ---
CONSERVAR_NUT = {
    "id_evaluacion": "Identifica la evaluación.", "grupo_hogar": "Hogar al que pertenece la evaluación.",
    "casa": "Casa dentro del hogar.", "peso_kg": "Medición completa.", "talla_cm": "Medición completa.",
    "imc_completado": "IMC final (reportado, calculado donde faltaba y corregido donde difería más de 0.11): completo.",
    "edad_meses_usada_para_tablas_ins": "Edad en meses desde el nacimiento hasta el mes de la evaluación: completa y es la edad confiable.",
}
ELIMINAR_NUT = {
    "id_persona": "Ya está en la columna id_persona del consolidado.", "periodo": "Ya está en la columna periodo del consolidado.",
    "imc_reportado": "Reemplazado por imc_completado (vacío en las 6 evaluaciones de marzo).",
    "imc_calculado": "Soporte del cálculo; el valor final está en imc_completado.",
    "diferencia_imc": "Soporte del cálculo; el valor final está en imc_completado.",
    "revision_imc": "Soporte del cálculo; el valor final está en imc_completado.",
    "imc_fue_calculado_por_nosotros": "Indicador del tratamiento (queda en el consolidado 1 y en la evidencia).",
    "imc_corregido_por_diferencia": "Indicador del tratamiento (queda en el consolidado 1 y en la evidencia).",
    "estado_mediciones": "Constante: todas las evaluaciones que continúan tienen peso y talla.",
    "fecha_nacimiento": "El nacimiento ya viene de la ficha nominal limpia.", "fecha_nacimiento_nominal": "Duplica el nacimiento de la ficha nominal.",
    "fecha_nacimiento_original": "Valor original (solo en el consolidado 1).",
    "fecha_evaluacion": "100 % vacía: el origen no trae fecha de evaluación (el mes viene del nombre del archivo).",
    "edad_reportada": "Edad escrita en el Excel: vacía en todo junio (venía como fórmula con TODAY) y adelantada; se reemplaza por la edad calculada.",
    "edad_formula_guardada": "Edad escrita en el Excel (fórmula con TODAY); se reemplaza por la edad calculada.",
    "edad_formula_original": "Fórmula original de edad (solo en el consolidado 1).",
    "edad_meses_texto_excel": "Edad escrita en el Excel, solo como contraste; se reemplaza por la edad calculada.",
    "sexo_nominal": "Vacío en las evaluaciones sin ficha nominal (agosto, etc.): se completa con el sexo del código (columna sexo).",
    "sexo_usado_para_tablas_ins": "Es la fuente del sexo completo; pasa a la columna sexo.",
    "sexo_archivo_origen": "Trazabilidad (solo en el consolidado 1).", "sexo_hoja_origen": "Trazabilidad (solo en el consolidado 1).",
    "sexo_fila_origen": "Trazabilidad (solo en el consolidado 1).",
    "estado_vinculo_nominal": "Ya está en estado_union.", "coincidencias_nominal_mes": "Trazabilidad de la unión (solo en el consolidado 1).",
    "identidad_nominal_observacion": "Constante vacía: los códigos con identidad contradictoria ya fueron excluidos.",
    "metodo_curva_original": "Describe la plantilla del hogar (solo en el consolidado 1).",
    "metodo_tabla_original": "Describe la plantilla del hogar (solo en el consolidado 1).",
    "indicador_imc_curva_original": "Describe la plantilla del hogar (solo en el consolidado 1).",
    "indicador_imc_tabla_original": "Describe la plantilla del hogar (solo en el consolidado 1).",
    "peso_original": "Valor original (solo en el consolidado 1).", "talla_original": "Valor original (solo en el consolidado 1).",
    "imc_original": "Valor original (solo en el consolidado 1).", "imc_formula_original": "Fórmula original (solo en el consolidado 1).",
    "perimetro_cefalico_original": "Solo existe en marzo (6 de 355 evaluaciones): no sirve para comparar el historial ni para agrupar.",
    "perimetro_cefalico_unidad": "Acompaña al perímetro cefálico, que solo existe en marzo.",
    "pa_original": "Solo existe en abril y su significado y unidad no están declarados: no sirve para el historial.",
    "pa_encabezado_original": "Acompaña a PA, que solo existe en abril.", "pa_unidad": "Acompaña a PA, que solo existe en abril.",
    "numero_original": "Número de fila de la hoja (solo en el consolidado 1).", "observacion_original": "Constante vacía: ya no hay evaluaciones con PROVINCIA.",
    "archivo_origen": "Trazabilidad (solo en el consolidado 1).", "hoja_origen": "Trazabilidad (solo en el consolidado 1).",
    "fila_origen": "Trazabilidad (solo en el consolidado 1).", "columnas_diagnostico_originales": "Trazabilidad (solo en el consolidado 1).",
    "asteriscos_diagnostico": "Control de calidad de la extracción (solo en el consolidado 1).",
    "vacios_diagnostico": "Control de calidad de la extracción (solo en el consolidado 1).",
    "indicadores_ausentes": "Control de calidad de la extracción (solo en el consolidado 1).",
    "normalizaciones_texto": "Control de calidad de la extracción (solo en el consolidado 1).",
    "diferencias_entre_metodos": "Control de calidad de la extracción (solo en el consolidado 1).",
    "observaciones_calidad": "Control de calidad de la extracción (solo en el consolidado 1).",
}

def decision_nutricion(col):
    if re.fullmatch(r"dx_(pe|te|pt|imc)_(curva|tabla)", col):
        return "CONSERVAR (vacío con significado)", ("Diagnóstico tratado. Los vacíos son '*' que no aplican por edad (P/E y P/T solo en menores de 5 años; "
                                                      "IMC desde los 5) o que no se pueden calcular; no se rellenan: una categoría no se promedia ni se inventa.")
    if re.fullmatch(r"dx_(pe|te|pt|imc)_(curva|tabla)_(original|estado|fue_calculado_por_nosotros)", col):
        return "ELIMINAR en consolidado 2", "Valor original, estado de la celda o indicador del tratamiento (solo en el consolidado 1)."
    if col in CONSERVAR_NUT:
        return "CONSERVAR", CONSERVAR_NUT[col]
    if col == "sexo_nominal":
        return "COMPLETAR", ELIMINAR_NUT[col]
    if col in ELIMINAR_NUT:
        return "ELIMINAR en consolidado 2", ELIMINAR_NUT[col]
    raise AssertionError("Falta decidir la columna de nutrición: " + col)

filas_eval = []
for col in nut.columns:
    s = nut[col]
    vacio = es_vacio(s)
    decision, motivo = decision_nutricion(col)
    fila = {"fuente": "nutrición", "columna": col, "registros": len(s), "vacios": int(vacio.sum()), "pct_vacio": round(100 * vacio.mean(), 1),
            "periodos_con_dato": f'{nut.loc[~vacio, "periodo"].nunique()} de {meses_totales}',
            "valores_distintos": s[~vacio].astype(str).nunique(), "decision": decision, "motivo": motivo}
    coincide = re.fullmatch(r"dx_(pe|te|pt|imc)_(curva|tabla)", col)
    if coincide:
        aplica = aplica_dx[coincide.group(1)]
        fila.update(registros_donde_aplica=int(aplica.sum()), vacios_donde_aplica=int((vacio & aplica).sum()),
                    pct_vacio_donde_aplica=round(100 * (vacio & aplica).sum() / aplica.sum(), 1))
    filas_eval.append(fila)

# Comprobaciones de las afirmaciones que sostienen las decisiones (si los datos cambian, el cuaderno avisa)
assert nut["fecha_evaluacion"].isna().all()
assert nut.loc[nut["perimetro_cefalico_original"].notna(), "periodo"].nunique() == 1
assert nut.loc[nut["pa_original"].notna(), "periodo"].nunique() == 1
assert nut["estado_mediciones"].nunique() == 1 and nut["observacion_original"].isna().all() and nut["identidad_nominal_observacion"].isna().all()
assert nut[["peso_kg", "talla_cm", "imc_completado", "edad_meses_usada_para_tablas_ins", "sexo_usado_para_tablas_ins"]].notna().all().all()

# --- Decisiones sobre las columnas de la ficha nominal (ya limpia, solo las fichas que continúan) ---
AUSENCIA_DECLARADA = {"no tiene", "no aplica", "ninguno", "na"}
limpia_continua = df_nominal_limpio[continua]
for posicion, col in enumerate(limpia_continua.columns):
    s = limpia_continua[col]
    vacio = es_vacio(s)
    dominante = s[~vacio].value_counts(normalize=True)
    if posicion in (POS_PERIODO, POS_ID):
        decision, motivo = "ELIMINAR en consolidado 2", "Ya está como periodo / id_persona en el consolidado."
    elif posicion == POS_SEXO:
        decision, motivo = "COMPLETAR", "Sin vacíos en la ficha; en las evaluaciones sin ficha nominal se completa con el sexo del código (columna sexo)."
    elif posicion == 3:
        decision, motivo = "CONSERVAR", "Pasa a la columna fecha_nacimiento."
    elif vacio.sum() == 0 and s.nunique() <= 1:
        decision, motivo = "ELIMINAR en consolidado 2", "Constante: todas las fichas tienen el mismo valor, no aporta."
    elif vacio.sum() > 0:
        decision, motivo = "CONSERVAR (vacío con significado)", ("Vacío porque el origen dice 'NA' / 'No tiene': es una ausencia declarada, no un dato perdido. "
                                                                  "No se rellena (inventar una fecha sería falso).")
    elif dominante.iloc[0] >= 0.99:
        decision, motivo = "CONSERVAR", "Casi constante (99 % o más en un solo valor): aporta poca variación, conviene revisarla antes de agrupar."
    else:
        decision, motivo = "CONSERVAR", "Sin vacíos."
    declarada = int(s.map(lambda v: isinstance(v, str) and v.casefold() in AUSENCIA_DECLARADA).sum())
    filas_eval.append({"fuente": "ficha nominal", "columna": col, "registros": len(s), "vacios": int(vacio.sum()),
                       "pct_vacio": round(100 * vacio.mean(), 1), "periodos_con_dato": "", "valores_distintos": s[~vacio].astype(str).nunique(),
                       "ausencia_declarada_en_texto": declarada, "decision": decision, "motivo": motivo})

evaluacion_vacios = pd.DataFrame(filas_eval)
columnas_nutricion_v2 = [c for c in nut.columns if decision_nutricion(c)[0].startswith("CONSERVAR")]
columnas_nominal_v2 = [c for c, d in zip(evaluacion_vacios.loc[evaluacion_vacios["fuente"] == "ficha nominal", "columna"],
                                         evaluacion_vacios.loc[evaluacion_vacios["fuente"] == "ficha nominal", "decision"])
                       if d.startswith("CONSERVAR")]

show_table(evaluacion_vacios.groupby(["fuente", "decision"]).size().rename("columnas"), "Decisión sobre cada columna")
show_table(evaluacion_vacios[(evaluacion_vacios["vacios"] > 0) | (evaluacion_vacios["decision"] == "COMPLETAR")]
           .set_index(["fuente", "columna"])[["registros", "vacios", "pct_vacio", "periodos_con_dato", "decision", "motivo"]],
           "Columnas con vacíos y qué se hace con ellas")

# Vacíos de diagnóstico que SÍ corresponden (el indicador aplica por edad pero no hay categoría) - detalle por celda
detalle_vacios = []
for indicador in ("pe", "te", "pt", "imc"):
    for bloque in ("curva", "tabla"):
        col = f"dx_{indicador}_{bloque}"
        falta = nut[col].isna() & aplica_dx[indicador]
        for _, r in nut[falta].iterrows():
            detalle_vacios.append({"id_evaluacion": r["id_evaluacion"], "id_persona": r["id_persona"], "periodo": r["periodo"],
                                   "columna": col, "estado_de_la_celda": r[col + "_estado"], "valor_original": r[col + "_original"],
                                   "edad_meses": r["edad_meses_usada_para_tablas_ins"], "archivo_origen": r["archivo_origen"],
                                   "hoja_origen": r["hoja_origen"], "fila_origen": r["fila_origen"]})
detalle_vacios = pd.DataFrame(detalle_vacios)
show_table(detalle_vacios.groupby(["columna", "estado_de_la_celda"]).size().rename("celdas"), "Diagnósticos vacíos donde el indicador sí aplica por edad")

# Cobertura por mes (porcentaje de evaluaciones con dato) y número de evaluaciones por NNA (insumo del Paso 4)
columnas_cobertura = ["peso_kg", "talla_cm", "imc_completado", "edad_meses_usada_para_tablas_ins"] + \
                     [f"dx_{i}_{b}" for i in ("pe", "te", "pt", "imc") for b in ("curva", "tabla")]
cobertura_por_mes = nut.groupby("periodo")[columnas_cobertura].agg(lambda s: round(100 * (~es_vacio(s)).mean(), 1))
cobertura_por_mes.insert(0, "evaluaciones", nut.groupby("periodo").size())
evaluaciones_por_nna = nut.groupby("id_persona").agg(evaluaciones=("id_evaluacion", "size"),
                                                     meses=("periodo", lambda s: ", ".join(sorted(s)))).reset_index()

,,celdas
columna,estado_de_la_celda,
dx_imc_curva,asterisco_original,2
dx_pe_curva,vacio_original,1
dx_pt_curva,asterisco_original,11
dx_te_curva,asterisco_original,2
dx_te_tabla,asterisco_original,1


In [ ]:
show_table(cobertura_por_mes, "Cobertura (% con dato) por mes")
show_table(evaluaciones_por_nna["evaluaciones"].value_counts().sort_index().rename("NNA").rename_axis("evaluaciones por NNA"),
           "NNA según el número de meses con evaluación")

export_evidence("12_evaluacion_vacios.xlsx", {
    "decision_por_columna": evaluacion_vacios,
    "dx_vacios_donde_aplica": detalle_vacios,
    "cobertura_por_mes": cobertura_por_mes.reset_index(),
    "evaluaciones_por_nna": evaluaciones_por_nna})

,evaluaciones,peso_kg,talla_cm,imc_completado,edad_meses_usada_para_tablas_ins,dx_pe_curva,dx_pe_tabla,dx_te_curva,dx_te_tabla,dx_pt_curva,dx_pt_tabla,dx_imc_curva,dx_imc_tabla
periodo,,,,,,,,,,,,,
2026-01,124,100.0,100.0,100.0,100.0,25.0,15.3,100.0,100.0,12.1,16.1,100.0,84.7
2026-02,7,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,0.0
2026-03,6,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,0.0
2026-04,93,100.0,100.0,100.0,100.0,40.9,18.3,98.9,100.0,12.9,19.4,81.7,81.7
2026-06,93,100.0,100.0,100.0,100.0,49.5,15.1,98.9,98.9,14.0,15.1,98.9,87.1
2026-08,32,100.0,100.0,100.0,100.0,0.0,0.0,100.0,100.0,0.0,0.0,100.0,100.0


,NNA
evaluaciones por NNA,
1,14
2,25
3,87
5,6


Evidencia guardada: C - KDD Steps/C - 3: Data Cleaning and Preprocessing/12_evaluacion_vacios.xlsx  (decision_por_columna: 118 filas, dx_vacios_donde_aplica: 17 filas, cobertura_por_mes: 6 filas, evaluaciones_por_nna: 132 filas)


**Comentario 16C:** (118 columnas: 85 de nutrición y 33 de la ficha nominal, el detalle de cada una está en `12_evaluacion_vacios.xlsx`).

| Decisión | Columnas | Ejemplos |
|---|---|---|
| **Eliminar en el consolidado 2** | 69 de nutrición, 4 nominales | `perimetro_cefalico_original` (solo marzo, 6 de 355), `pa_original` (solo abril, sin unidad ni significado), `fecha_evaluacion` (100 % vacía), `edad_reportada` (vacía en todo junio y adelantada), `imc_reportado` (vacío en marzo), todo lo `_original` / `_estado` y la trazabilidad; `Comunicado_DGA` y `Juzgado/UPE` (constantes) |
| **Completar** | 1 de nutrición (+ `Sexo`) | `sexo_nominal` está vacío en 47 evaluaciones sin ficha nominal del mes (agosto, etc.): se completa con el sexo del código, que es único porque los códigos contradictorios ya se excluyeron |
| **Conservar (vacío con significado)** | 8 de nutrición, 3 nominales | los 8 diagnósticos `dx_…` y las fechas de resolución, oficio y PTI |
| **Conservar** | el resto | peso, talla, `imc_completado`, edad en meses calculada, hogar, casa y las columnas nominales sin vacíos |

**Por qué no se rellena con promedios de meses anteriores:** peso, talla e IMC no tienen ningún vacío, así que no hay nada que rellenar, los diagnósticos son categorías, que no se promedian, y las fechas vacías son «NA» / «No tiene» declarados, e inventarlas sería falso. Tampoco se rellenan meses sin evaluación, ya que la mayoría de los NNA tiene 3 evaluaciones (87 de 132, 14 tienen solo 1, 25 tienen 2 y 6 tienen 5), y rellenar meses ausentes reduciría artificialmente la variación del historial.

**Los diagnósticos con muchos vacíos son estructurales, no datos perdidos:** P/E y P/T solo aplican a menores de 5 años (63 de 355 evaluaciones), por eso `dx_pe_*` y `dx_pt_*` están vacíos en 64 %–85 %. Donde sí aplican por edad faltan solo 17 celdas (11 de ellas P/T del bloque OMS, que no se puede calcular). `dx_imc_tabla` no existe en las plantillas de febrero y marzo. **Para el Paso 4**, P/E y P/T solo sirven como variables del subgrupo menor de 5 años, si se quiere agrupar a todos los NNA, conviene no usarlas.

**Evidencia:** `12_evaluacion_vacios.xlsx` (decisión y motivo por columna, diagnósticos vacíos donde sí aplican, cobertura por mes y evaluaciones por NNA).

# 8. Unión de la ficha nominal con la nutrición tratada

Se unen registros cuando coinciden identificador y mes, existe una única ficha nominal candidata, coincide el nacimiento y el código no tiene alertas de identidad. Si no se cumple, ambas fuentes quedan como filas separadas

In [ ]:
# Se agregan al orden de columnas las nuevas columnas de este notebook (las originales no se tocan)
columnas_nuevas = ["imc_completado", "imc_fue_calculado_por_nosotros", "edad_meses_texto_excel",
                   "dx_pe_tabla_fue_calculado_por_nosotros", "dx_te_tabla_fue_calculado_por_nosotros",
                   "dx_pt_tabla_fue_calculado_por_nosotros", "dx_imc_tabla_fue_calculado_por_nosotros",
                   "sexo_usado_para_tablas_ins", "edad_meses_usada_para_tablas_ins"]
nutrition_columns = nutrition_columns + [c for c in columnas_nuevas if c not in nutrition_columns]
nutrition_df = nutrition_df.reindex(columns=nutrition_columns)
print("Columnas de nutrición:", len(nutrition_columns))

Columnas de nutrición: 85


In [ ]:
# Diccionario para guardar las evaluaciones vinculadas
linked_nutrition = {}

# Lista para guardar las evaluaciones sin vinculación segura
unlinked_nutrition = []

# Convertir los registros nutricionales a diccionarios
nutrition_records = (
    nutrition_df.astype(object)
    .where(pd.notna(nutrition_df), None)
    .to_dict("records")
)

for nutrition_record in nutrition_records:

    # Buscar coincidencia por periodo e identificador
    key = (
        nutrition_record["periodo"],
        nutrition_record["id_persona"]
    )

    candidates = nominal_by_key.get(key, [])

    # Verificar que la vinculación sea segura
    safe_link = (
        len(candidates) == 1
        and candidates[0]["dob"] is not None
        and candidates[0]["dob"] == nutrition_record["fecha_nacimiento"]
        and nutrition_record["id_persona"] not in conflicting_ids
    )

    # Comprobar consistencia con la validación anterior
    assert safe_link == (
        nutrition_record["estado_vinculo_nominal"]
        == "unico_mes_y_nacimiento_coinciden"
    )

    if safe_link:

        # Recuperar la fila original de la ficha nominal
        nominal_row_number = candidates[0]["fila"]

        # Evitar asociar más de una evaluación a una misma ficha
        assert nominal_row_number not in linked_nutrition, (
            "Más de una evaluación para la misma ficha."
        )

        linked_nutrition[nominal_row_number] = nutrition_record

    else:
        # Conservar por separado las evaluaciones no vinculadas
        unlinked_nutrition.append(nutrition_record)

# Mostrar el resultado
print("Resultado de los criterios de vinculación")

display(
    nutrition_df["estado_vinculo_nominal"]
    .value_counts()
    .rename("evaluaciones")
)

Resultado de los criterios de vinculación


,evaluaciones
estado_vinculo_nominal,
unico_mes_y_nacimiento_coinciden,318
periodo_nominal_no_disponible,32
id_ausente_en_mes_nominal,5


**Comentario 17:** De las 355 evaluaciones, 308 se vinculan con seguridad, las otras 47 no se vinculan: 32 son de agosto (la ficha nominal termina en julio), 10 tienen una fecha de nacimiento distinta a la de la ficha (PPA_FN_0004, 0028, 0066 y 0064) y 5 corresponden a un código ausente de la ficha de ese mes. Ya no hay evaluaciones «de identidad contradictoria», porque esos códigos fueron excluidos antes. Se conserva la comprobación de que un vínculo seguro nunca asigne más de una evaluación a la misma ficha.

Se realiza la unión final de la ficha nominal con las evaluaciones nutricionales, integrando ambas fuentes en un único DataFrame (unified_df). Para ello, se conserva los registros nominales y se incorpora la información nutricional cuando existe una coincidencia segura entre el identificador, el periodo y la fecha de nacimiento. Las evaluaciones que no cumplen los criterios de vinculación se mantienen como registros independientes para evitar asociaciones incorrectas. Además, se conserva los datos originales, la procedencia de cada registro y las observaciones de calidad, permitiendo identificar si una fila contiene información de ambas fuentes, solo de la ficha nominal o solo de nutrición. Finalmente, se genera tablas para verificar los resultados de la consolidación.

In [ ]:
# Preparar los registros nominales desde el DataFrame df
nominal_records = []

for fila, row in df_nominal.reset_index(drop=True).iterrows():
    registro = {
        "fila": fila + 2,
        "periodo": clean_text(row.iloc[0]).replace("_", "-"),
        "id": clean_text(row.iloc[1]),
        "sexo": clean_text(row.iloc[2]),
        "dob": parse_birth_date(row.iloc[3]),
        "values": [serialize_original(valor) for valor in row]
    }

    nominal_records.append(registro)

len(nominal_records)

# Las fichas de los códigos inconsistentes no forman parte del consolidado
nominal_records_todos = nominal_records
nominal_records = [r for r in nominal_records_todos if r["id"] not in conflicting_ids]
print("Fichas nominales leídas:", len(nominal_records_todos), "| que se usan en el consolidado:", len(nominal_records))
assert len(nominal_records_todos) - len(nominal_records) == len(nominal_excluida)

Fichas nominales leídas: 2667 | que se usan en el consolidado: 2498


In [ ]:
issues_by_evaluation = defaultdict(list)
for observation in issues:
    issues_by_evaluation[observation["id_evaluacion"]].append(observation)
notes_by_sheet = defaultdict(list)
for note in notes:
    notes_by_sheet[(note["archivo_origen"], note["hoja_origen"])].append(note)
nutrition_keys = {(r["periodo"], r["id_persona"]) for r in nutrition_records}
common_columns = [
    "id_registro", "id_persona", "periodo", "origen_registro", "estado_union",
    "identidad_nominal_observacion", "nominal_repeticiones_id_mes",
    "nominal_fecha_nacimiento", "nominal_sexo", "nutricion_excluida_motivo"
]
first_nutrition_columns = nutrition_columns[:30]
unified_columns = (
    common_columns + ["nut_" + k for k in first_nutrition_columns] +
    ["nom_" + str(header) for header in nominal_headers] +
    ["nut_" + k for k in nutrition_columns if k not in first_nutrition_columns] +
    ["nom_archivo_origen", "nom_hoja_origen", "nom_fila_origen",
     "nut_detalle_observaciones", "nut_notas_hoja_originales", "nut_tolerancia_revision_imc"]
)

def build_unified_record(nominal_record=None, nutrition_record=None):
    n, u = nominal_record, nutrition_record
    person_id = n["id"] if n else u["id_persona"]
    period = n["periodo"] if n else u["periodo"]
    record = dict.fromkeys(unified_columns)
    record.update(
        id_registro=f'NOM_{n["fila"]:05d}' if n else u["id_evaluacion"],
        id_persona=person_id, periodo=period,
        origen_registro="nominal_y_nutricion" if n and u else "solo_nominal" if n else "solo_nutricion",
        estado_union=("coincidencia_segura" if n and u else u["estado_vinculo_nominal"] if u
                      else "sin_nutricion_en_periodo" if (period, person_id) not in nutrition_keys
                      else "nutricion_no_vinculada_por_identidad"),
        identidad_nominal_observacion=("identificador_con_datos_contradictorios"
                                       if person_id in conflicting_ids else None),
        nominal_repeticiones_id_mes=len(nominal_by_key.get((period, person_id), [])),
        nutricion_excluida_motivo="PROVINCIA" if (period, person_id) in provincia_keys else None
    )
    if n:
        record.update(nominal_fecha_nacimiento=n["dob"], nominal_sexo=n["sexo"],
                      nom_archivo_origen=nominal_path.name, nom_hoja_origen=nominal_sheet_name,
                      nom_fila_origen=n["fila"])
        record.update({"nom_" + header: value for header, value in zip(nominal_headers, n["values"])})
    if u:
        record.update({"nut_" + key: u[key] for key in nutrition_columns})
        observations = issues_by_evaluation[u["id_evaluacion"]]
        if observations:
            record["nut_detalle_observaciones"] = json.dumps(observations, ensure_ascii=False)
        sheet_observations = notes_by_sheet[(u["archivo_origen"], u["hoja_origen"])]
        if sheet_observations:
            record["nut_notas_hoja_originales"] = json.dumps(sheet_observations, ensure_ascii=False)
        record["nut_tolerancia_revision_imc"] = IMC_REVIEW_TOLERANCE
    return record

unified_records = [build_unified_record(n, linked_nutrition.get(n["fila"])) for n in nominal_records]
unified_records.extend(build_unified_record(nutrition_record=u) for u in unlinked_nutrition)
unified_df = pd.DataFrame(unified_records, columns=unified_columns).sort_values(
    ["periodo", "id_persona", "origen_registro", "id_registro"]
).reset_index(drop=True)
show_table(unified_df["origen_registro"].value_counts().rename("registros"), "Origen de las filas resultantes")
show_table(unified_df["estado_union"].value_counts().rename("registros"), "Estados de la unión")
show_table(unified_df[["id_registro", "id_persona", "periodo", "origen_registro", "estado_union"]].head(8),
           "Muestra del conjunto preparado")

,registros
origen_registro,
solo_nominal,2180
nominal_y_nutricion,318
solo_nutricion,37


,registros
estado_union,
sin_nutricion_en_periodo,2180
coincidencia_segura,318
periodo_nominal_no_disponible,32
id_ausente_en_mes_nominal,5


,id_registro,id_persona,periodo,origen_registro,estado_union
0,NOM_00002,PPA_FN_0001,2025-01,solo_nominal,sin_nutricion_en_periodo
1,NOM_00003,PPA_FN_0002,2025-01,solo_nominal,sin_nutricion_en_periodo
2,NOM_00004,PPA_FN_0003,2025-01,solo_nominal,sin_nutricion_en_periodo
3,NOM_00005,PPA_FN_0004,2025-01,solo_nominal,sin_nutricion_en_periodo
4,NOM_00006,PPA_FN_0005,2025-01,solo_nominal,sin_nutricion_en_periodo
5,NOM_00007,PPA_FN_0006,2025-01,solo_nominal,sin_nutricion_en_periodo
6,NOM_00008,PPA_FN_0007,2025-01,solo_nominal,sin_nutricion_en_periodo
7,NOM_00009,PPA_FN_0008,2025-01,solo_nominal,sin_nutricion_en_periodo


**Comentario 18:** El resultado tiene 2 545 filas: 308 con ficha nominal y nutrición, 2 190 solo con ficha nominal (de las cuales 23 no tienen nutrición porque ese mes fue `PROVINCIA`) y 47 solo con nutrición. La cuenta cuadra: 2 498 fichas + 355 evaluaciones − 308 vínculos = 2 545. Los datos se identifican por prefijo: `nom_` para la ficha nominal y `nut_` para la nutrición, más las columnas de identificación, estado de unión y procedencia, de modo que cada fila puede rastrearse hasta su celda de origen.

Las filas solo nominales (2 190) corresponden sobre todo a los meses de la ficha sin evaluación nutricional (la nutrición solo cubre seis meses). Las solo nutricionales (47) se mantienen como filas independientes para no crear asociaciones incorrectas.

# 9. Verificación y salida

In [ ]:
# Cada fila de origen debe seguir representada una sola vez.
nominal_output = unified_df[unified_df["nom_fila_origen"].notna()]
nutrition_output = unified_df[unified_df["nut_id_evaluacion"].notna()]

assert set(nominal_output["nom_fila_origen"]) == {r["fila"] for r in nominal_records}
assert not nominal_output["nom_fila_origen"].duplicated().any()
assert set(nutrition_output["nut_id_evaluacion"]) == set(nutrition_df["id_evaluacion"])
assert not nutrition_output["nut_id_evaluacion"].duplicated().any()
assert unified_df["id_registro"].is_unique
assert not unified_df["id_registro"].isna().any()
assert not unified_df["id_persona"].isna().any()
assert unified_df.columns.is_unique
assert len(unified_df) == len(nominal_records) + len(nutrition_df) - len(linked_nutrition)

# Comprobar los 33 campos nominales contra cada fila recibida.
nominal_lookup = nominal_output.set_index("nom_fila_origen")
for record in nominal_records:
    for header, expected in zip(nominal_headers, record["values"]):
        actual = nominal_lookup.loc[record["fila"], "nom_" + header]
        assert (pd.isna(actual) and expected is None) or actual == expected

# Comprobar mediciones y fórmulas contra las celdas nutricionales de origen.
nutrition_lookup = nutrition_df_bruto.set_index(["archivo_origen", "hoja_origen", "fila_origen"])
source_field_checks = 0
for filename, sheets in nutrition_sources.items():
    for sheet in sheets:
        headers = sheet["rows"][2]
        weight_index = next(i for i, h in enumerate(headers) if h and str(h).startswith("PESO"))
        height_index = next(i for i, h in enumerate(headers) if h and str(h).startswith("TALLA"))
        for row_number, values in enumerate(sheet["rows"], start=1):
            if len(values) < 2 or not re.fullmatch(r"PPA_FN_\d+", str(values[1])):
                continue
            target = nutrition_lookup.loc[(filename, sheet["sheet"], row_number)]
            for field, index in [("id_persona", 1), ("peso_original", weight_index),
                                 ("talla_original", height_index), ("fecha_nacimiento_original", 2)]:
                expected = serialize_original(values[index])
                actual = target[field]
                assert (pd.isna(actual) and expected is None) or actual == expected
                source_field_checks += 1

controls = pd.DataFrame([
    ("Fichas nominales conservadas",
     len(nominal_output),
     len(nominal_records)),

    ("Evaluaciones nutricionales conservadas",
     len(nutrition_output),
     len(nutrition_df)),

    ("Vínculos entre fuentes",
     len(linked_nutrition),
     None),

    ("Registros unificados",
     len(unified_df),
     len(nominal_records) + len(nutrition_df) - len(linked_nutrition)),

    ("Columnas del conjunto",
     unified_df.shape[1],
     len(unified_columns)),

    ("Identificadores de registro distintos",
     unified_df["id_registro"].nunique(),
     len(unified_df)),

    ("IMC calculados",
     nutrition_df["imc_calculado"].notna().sum(),
     None),

    ("IMC señalados para revisión",
     (nutrition_df["revision_imc"] == "revisar").sum(),
     None)

], columns=["control", "resultado", "esperado"]).set_index("control")


# Evaluar solamente los controles con un valor esperado definido
controls["coincide"] = (
    controls["esperado"].isna() |
    controls["resultado"].eq(controls["esperado"])
)

assert controls["coincide"].all()
show_table(controls, "Conservación y conciliación de registros")

# Controles específicos de los tratamientos de este notebook
assert not unified_df["id_persona"].isin(conflicting_ids).any(), "Quedó un código inconsistente en el consolidado."
assert not (unified_df["nut_estado_mediciones"] == "sin_medicion_provincia").any(), "Quedó una evaluación PROVINCIA."
for indicador in ("pe", "te", "pt", "imc"):
    marcada = unified_df[f"nut_dx_{indicador}_tabla_fue_calculado_por_nosotros"].fillna(False).astype(bool)
    assert unified_df.loc[marcada, f"nut_dx_{indicador}_tabla_original"].eq("*").all(), indicador
print("Controles de exclusión y de flags superados.")

,resultado,esperado,coincide
control,,,
Fichas nominales conservadas,2498,2498.0,True
Evaluaciones nutricionales conservadas,355,355.0,True
Vínculos entre fuentes,318,—,True
Registros unificados,2535,2535.0,True
Columnas del conjunto,134,134.0,True
Identificadores de registro distintos,2535,2535.0,True
IMC calculados,355,—,True
IMC señalados para revisión,9,—,True


Controles de exclusión y de flags superados.


/tmp/ipykernel_1647/1553224293.py:90: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  marcada = unified_df[f"nut_dx_{indicador}_tabla_fue_calculado_por_nosotros"].fillna(False).astype(bool)
/tmp/ipykernel_1647/1553224293.py:90: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  marcada = unified_df[f"nut_dx_{indicador}_tabla_fue_calculado_por_nosotros"].fillna(False).astype(bool)
/tmp/ipykernel_1647/1553224293.py:90: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_obj

**Comentario 19:** Cada ficha y cada evaluación que continúa aparece una sola vez, los 33 campos nominales y los valores originales de nutrición coinciden celda por celda con el Excel de origen (incluso para las evaluaciones excluidas, que se verifican contra la copia sin tratar), los identificadores de registro son únicos y la cuenta de filas y columnas cuadra. El consolidado tiene 133 columnas: las 123 del esquema anterior más 10 nuevas (marca de exclusión por `PROVINCIA`, IMC completado y su indicador, edad escrita para contraste, edad y sexo usados en las tablas, y los 4 indicadores de diagnóstico calculado).

# 9A. Columnas limpias en el consolidado 1

El consolidado 1 conserva las 33 columnas nominales tal como llegaron (`nom_<columna>`) y agrega a su lado la versión limpia (`nom_<columna>_limpio`), más `sexo_completado`.

In [ ]:
# COLUMNAS LIMPIAS EN EL CONSOLIDADO 1
# El consolidado conserva las 33 columnas nominales tal como llegaron (nom_<columna>) y ahora agrega a su lado la versión limpia
# (nom_<columna>_limpio): sin comillas, con fechas reales y con el texto homologado. El original no se toca.
# También agrega sexo_completado: el sexo de la ficha y, donde no hay ficha del mes (evaluaciones de agosto, etc.), el sexo del código.

columnas_limpias = ["nom_" + str(h) + "_limpio" for h in nominal_headers]
limpio_por_fila = df_nominal_limpio.copy()
limpio_por_fila.index = limpio_por_fila.index + 2                      # número de fila en el Excel de origen
limpio_por_fila.columns = columnas_limpias

filas_origen = unified_df["nom_fila_origen"].map(lambda f: int(f) if pd.notna(f) else -1)
bloque_limpio = limpio_por_fila.reindex(filas_origen.values)           # filas solo nutrición -> vacío
bloque_limpio.index = unified_df.index

sexo_completado = unified_df["nominal_sexo"].where(unified_df["nominal_sexo"].notna(), unified_df["nut_sexo_usado_para_tablas_ins"]).rename("sexo_completado")
columnas_antes = unified_df.shape[1]
unified_df = pd.concat([unified_df, bloque_limpio, sexo_completado], axis=1)

# Controles: cada fila nominal lleva su ficha limpia, las filas solo nutrición no inventan datos nominales
assert unified_df.shape[1] == columnas_antes + len(columnas_limpias) + 1 and unified_df.columns.is_unique
con_ficha = unified_df["nom_fila_origen"].notna()
assert unified_df.loc[~con_ficha, columnas_limpias].isna().all().all()
filas_presentes = set(unified_df["nom_fila_origen"].dropna().astype(int))
for fila in sorted(filas_presentes)[::250]:
    esperado = limpio_por_fila.loc[fila]
    actual = unified_df.loc[unified_df["nom_fila_origen"] == fila, columnas_limpias].iloc[0]
    assert all((pd.isna(a) and pd.isna(e)) or a == e for a, e in zip(actual, esperado)), fila
assert unified_df["sexo_completado"].notna().all(), "Hay filas sin sexo."
assert unified_df["sexo_completado"].isin(["M", "F"]).all()
print("Consolidado 1:", unified_df.shape[0], "filas x", unified_df.shape[1], "columnas (", len(columnas_limpias), "limpias + sexo_completado nuevas )")
print("Sexo completado en filas sin ficha nominal:", int((unified_df["nominal_sexo"].isna() & unified_df["sexo_completado"].notna()).sum()))
show_table(unified_df[["id_registro", "nom_Sexo", "nom_Sexo_limpio", "nom_Fecha de nacimiento", "nom_Fecha de nacimiento_limpio",
                       "nom_Tipo de documento", "nom_Tipo de documento_limpio"]].head(5).set_index("id_registro"),
           "Muestra: original y limpio, lado a lado")

Consolidado 1: 2535 filas x 168 columnas ( 33 limpias + sexo_completado nuevas )
Sexo completado en filas sin ficha nominal: 37


,nom_Sexo,nom_Sexo_limpio,nom_Fecha de nacimiento,nom_Fecha de nacimiento_limpio,nom_Tipo de documento,nom_Tipo de documento_limpio
id_registro,,,,,,
NOM_00002,"""F""",F,"""2022-03-26""",2022-03-26,"""Resolución Admi.""",Resolución Admi.
NOM_00003,"""M""",M,"""2021-11-30""",2021-11-30,"""Resolución Admi.""",Resolución Admi.
NOM_00004,"""M""",M,"""2022-03-28""",2022-03-28,"""Resolución Admi.""",Resolución Admi.
NOM_00005,"""M""",M,"""2023-09-01""",2023-09-01,"""Resolución Admi.""",Resolución Admi.
NOM_00006,"""F""",F,"""2021-11-06""",2021-11-06,"""Resolución Admi.""",Resolución Admi.


**Comentario 19A** Se agregan 34 columnas (33 limpias y `sexo_completado`) a las 134 anteriores: **2 545 filas × 168 columnas**. El original no se modificó, y por eso los controles del Comentario 19 (que comparan cada campo nominal con el Excel de origen) siguen valiendo. Las 47 filas solo de nutrición no llevan datos nominales limpios (no hay ficha), pero sí `sexo_completado`.

In [ ]:
# Preparar valores de salida: fechas reales y fórmulas originales como texto.
def prepare_export_frame(frame, date_columns):
    result = frame.copy()
    for column in date_columns:
        result[column] = pd.to_datetime(result[column], format="%Y-%m-%d", errors="raise")
    for column in result.columns:
        if result[column].dtype == "object" or pd.api.types.is_string_dtype(result[column]):
            result[column] = result[column].map(
                lambda value: "'" + value if isinstance(value, str) and value.startswith("=") else value
            )
    return result

nutrition_export = prepare_export_frame(nutrition_df, [
    "fecha_nacimiento", "fecha_nacimiento_nominal", "fecha_evaluacion"
])
unified_export = prepare_export_frame(unified_df, [
    "nominal_fecha_nacimiento", "nut_fecha_nacimiento",
    "nut_fecha_nacimiento_nominal", "nut_fecha_evaluacion"
])

def compare_frames(actual, expected, key):
    assert set(actual.columns) == set(expected.columns), "El esquema no coincide."
    assert len(actual) == len(expected), "El número de registros no coincide."
    assert actual[key].is_unique and expected[key].is_unique
    left = actual.set_index(key).sort_index()
    right = expected.set_index(key).sort_index().reindex(columns=left.columns)
    assert left.index.equals(right.index), "Los identificadores de registro no coinciden."
    differences = []
    for column in left.columns:
        for record_id, a, b in zip(left.index, left[column], right[column]):
            a_missing, b_missing = pd.isna(a), pd.isna(b)
            if a_missing and b_missing:
                continue
            if a_missing != b_missing:
                differences.append((record_id, column, str(a), str(b)))
            elif isinstance(a, (int, float, np.number)) and isinstance(b, (int, float, np.number)):
                if not np.isclose(a, b, rtol=1e-10, atol=1e-9):
                    differences.append((record_id, column, str(a), str(b)))
            elif "formula_original" in column:
                a_text = a[1:] if isinstance(a, str) and a.startswith("'=") else a
                b_text = b[1:] if isinstance(b, str) and b.startswith("'=") else b
                if a_text != b_text:
                    differences.append((record_id, column, str(a), str(b)))
            elif a != b:
                differences.append((record_id, column, str(a), str(b)))
    return pd.DataFrame(differences, columns=[key, "campo", "reconstruido", "consolidado"])

In [ ]:
# GENERAR LOS CONSOLIDADOS
# Siempre se regeneran: sus valores ya incluyen los tratamientos de este notebook (exclusiones, normalización, IMC y diagnósticos).
consolidated_dir.mkdir(parents=True, exist_ok=True)
with pd.ExcelWriter(nutrition_path, engine="openpyxl", datetime_format="DD/MM/YYYY") as writer:
    nutrition_export.to_excel(writer, sheet_name="Nutricion", index=False)
    normalization_summary.to_excel(writer, sheet_name="Normalizaciones", index=False)
    sheet_notes.to_excel(writer, sheet_name="Notas de hojas", index=False)
    quality_issues.to_excel(writer, sheet_name="Observaciones", index=False)
unified_export.to_excel(unified_path, sheet_name="Dataset", index=False, engine="openpyxl")

# Verificar que lo guardado coincide con lo que está en memoria
for path, sheet, frame, key in [
    (nutrition_path, "Nutricion", nutrition_export, "id_evaluacion"),
    (unified_path, "Dataset", unified_export, "id_registro")
]:
    saved = pd.read_excel(path, sheet_name=sheet, keep_default_na=False).replace("", None)
    assert compare_frames(frame, saved, key).empty

# Los archivos de entrada no deben haber cambiado durante el proceso
assert all(hashlib.sha256((path).read_bytes()).hexdigest() == source_fingerprints[path.name]
           for path in nominal_files + nutrition_files), "Un archivo de entrada cambió."

show_table(pd.DataFrame([
    {"archivo": nutrition_path.name, "ubicación": str(nutrition_path.relative_to(project_dir)),
     "registros": len(nutrition_export), "columnas_de_datos": nutrition_export.shape[1]},
    {"archivo": unified_path.name, "ubicación": str(unified_path.relative_to(project_dir)),
     "registros": len(unified_export), "columnas_de_datos": unified_export.shape[1]}
]).set_index("archivo"), "Conjuntos disponibles para el análisis")

,ubicación,registros,columnas_de_datos
archivo,,,
Nutricion_consolidada.xlsx,Nutricion_consolidada.xlsx,355,85
Dataset_unificado.xlsx,Dataset_unificado.xlsx,2535,168


**Comentario 20:** El consolidado 1 queda con **168 columnas** (las 133 anteriores, `imc_corregido_por_diferencia`, las 33 columnas nominales limpias `nom_<columna>_limpio` y `sexo_completado`) y se agrega `Dataset_unificado_2.xlsx` (consolidado 2, 50 columnas, solo valores tratados).

# 9B. Consolidado 2: solo los valores tratados

`Dataset_unificado_2.xlsx` se arma desde el consolidado 1 con las decisiones de la evaluación de vacíos.

In [ ]:
# CONSOLIDADO 2: SOLO LOS VALORES TRATADOS (sin columnas originales ni de trazabilidad)
# Se arma desde el consolidado 1 con las decisiones de la evaluación de vacíos. Una sola columna por variable:
#  - llaves: id_registro, id_persona, periodo, origen_registro, estado_union y nutricion_excluida_motivo;
#  - sexo (completado) y fecha_nacimiento (limpia), que reemplazan a las columnas duplicadas de ambas fuentes;
#  - las columnas nominales limpias que se conservan (nom_<columna>) y las de nutrición tratadas que se conservan (nut_<columna>).

unified_2_path = consolidated_dir / "Dataset_unificado_2.xlsx"
columnas_nominal_v2_ = [c for c in columnas_nominal_v2 if c != nominal_headers[3]]       # el nacimiento va como fecha_nacimiento
columnas_nutricion_v2_ = [c for c in columnas_nutricion_v2 if c != "sexo_usado_para_tablas_ins"]   # va como sexo

unified_2 = unified_df[["id_registro", "id_persona", "periodo", "origen_registro", "estado_union", "nutricion_excluida_motivo"]].copy()
unified_2["sexo"] = unified_df["sexo_completado"]
unified_2["fecha_nacimiento"] = unified_df["nom_" + nominal_headers[3] + "_limpio"]
for columna in columnas_nominal_v2_:
    unified_2["nom_" + columna] = unified_df["nom_" + columna + "_limpio"]
for columna in columnas_nutricion_v2_:
    unified_2["nut_" + columna] = unified_df["nut_" + columna]

# Controles del consolidado 2
assert len(unified_2) == len(unified_df) and unified_2["id_registro"].is_unique and unified_2.columns.is_unique
assert not [c for c in unified_2.columns if c.endswith(("_original", "_estado", "_limpio")) or "_fue_calculado" in c], "Quedó una columna original."
assert unified_2["sexo"].notna().all() and unified_2["id_persona"].notna().all()
columnas_texto_v2 = [c for c in unified_2.columns if unified_2[c].dtype == "object"]
assert not unified_2[columnas_texto_v2].astype(str).apply(lambda s: s.str.contains('"')).any().any(), "Quedaron comillas."
assert not unified_2.isna().all().any(), "Hay columnas totalmente vacías."
assert set(unified_2["nut_id_evaluacion"].dropna()) == set(nutrition_df["id_evaluacion"])

# Guardar y verificar que lo guardado coincide con lo que está en memoria
unified_2_export = prepare_export_frame(unified_2, [])
with pd.ExcelWriter(unified_2_path, engine="openpyxl", datetime_format="DD/MM/YYYY", date_format="DD/MM/YYYY") as writer:
    unified_2_export.to_excel(writer, sheet_name="Dataset", index=False)
saved_2 = pd.read_excel(unified_2_path, sheet_name="Dataset", keep_default_na=False).replace("", None)
assert compare_frames(unified_2_export, saved_2, "id_registro").empty

# Vacíos que quedan en el consolidado 2 (por tipo de fila), para decidir en el Paso 4
vacios_c2 = pd.DataFrame({"pct_vacio_total": unified_2.isna().mean().mul(100).round(1)})
for origen, grupo in unified_2.groupby("origen_registro"):
    vacios_c2["pct_vacio_" + origen] = grupo.isna().mean().mul(100).round(1)
vacios_c2 = vacios_c2.reset_index().rename(columns={"index": "columna"})
show_table(pd.DataFrame([{"archivo": unified_2_path.name, "ubicación": str(unified_2_path.relative_to(project_dir)),
                          "registros": len(unified_2_export), "columnas": unified_2_export.shape[1]}]).set_index("archivo"),
           "Consolidado 2 (solo valores tratados)")
show_table(vacios_c2[vacios_c2["pct_vacio_total"] > 0].set_index("columna"), "Vacíos que quedan en el consolidado 2 (% por tipo de fila)")
export_evidence("13_vacios_consolidado_2.xlsx", {"vacios_por_columna": vacios_c2, "columnas_del_consolidado_2": pd.DataFrame({"columna": unified_2.columns})})

# Los archivos de entrada siguen sin cambios
assert all(hashlib.sha256((path).read_bytes()).hexdigest() == source_fingerprints[path.name]
           for path in nominal_files + nutrition_files), "Un archivo de entrada cambió."

,ubicación,registros,columnas
archivo,,,
Dataset_unificado_2.xlsx,Dataset_unificado_2.xlsx,2535,50


,pct_vacio_total,pct_vacio_nominal_y_nutricion,pct_vacio_solo_nominal,pct_vacio_solo_nutricion
columna,,,,
nutricion_excluida_motivo,99.1,100.0,98.9,100.0
fecha_nacimiento,1.5,0.0,0.0,100.0
nom_Departamento,1.5,0.0,0.0,100.0
nom_Provincia,1.5,0.0,0.0,100.0
nom_Distrito,1.5,0.0,0.0,100.0
nom_Nacionalidad,1.5,0.0,0.0,100.0
nom_Fecha de ingreso,1.5,0.0,0.0,100.0
nom_Observaciones y/o comentarios de la forma de ingreso de la NNA,1.5,0.0,0.0,100.0
nom_Tipo de documento,1.5,0.0,0.0,100.0


Evidencia guardada: C - KDD Steps/C - 3: Data Cleaning and Preprocessing/13_vacios_consolidado_2.xlsx  (vacios_por_columna: 50 filas, columnas_del_consolidado_2: 50 filas)


**Comentario 20A:** Tiene las mismas 2 545 filas y 50 columnas: 6 llaves (`id_registro`, `id_persona`, `periodo`, `origen_registro`, `estado_union`, `nutricion_excluida_motivo`), `sexo` (completado), `fecha_nacimiento` (limpia), 27 columnas nominales limpias (`nom_…`) y 15 de nutrición tratadas (`nut_…`: hogar, casa, peso, talla, IMC final, edad en meses y los 8 diagnósticos). No quedan columnas `_original`, `_estado` ni indicadores de tratamiento, se verifica que no haya comillas, que ninguna columna esté totalmente vacía y que lo guardado coincida con lo que está en memoria. La trazabilidad (archivo, hoja y fila de origen) sigue en el consolidado 1.

Los vacíos que quedan en este consolidado se resumen en `13_vacios_consolidado_2.xlsx`

In [ ]:
# EVIDENCIA DE LAS SEPARACIONES DE LA UNIÓN Y RESUMEN FINAL
# Evaluaciones de nutrición que no quedaron unidas a una ficha nominal (se conservan como filas "solo nutrición")
solo_nutricion = unified_df[unified_df["origen_registro"] == "solo_nutricion"]
hojas_sin_union = {"resumen": solo_nutricion["estado_union"].value_counts().rename("registros").reset_index()}
for estado, grupo in solo_nutricion.groupby("estado_union"):
    hojas_sin_union[estado[:31]] = grupo[["id_registro", "id_persona", "periodo", "nut_fecha_nacimiento",
                                          "nut_fecha_nacimiento_nominal", "nut_archivo_origen", "nut_hoja_origen", "nut_fila_origen"]]
export_evidence("08_nutricion_sin_union_con_ficha_nominal.xlsx", hojas_sin_union)

sin_nutricion_por_provincia = unified_df[unified_df["nutricion_excluida_motivo"] == "PROVINCIA"]
export_evidence("09_fichas_sin_nutricion_por_provincia.xlsx", {
    "fichas_nominales_mes_sin_evaluacion": sin_nutricion_por_provincia[["id_registro", "id_persona", "periodo", "estado_union", "nutricion_excluida_motivo"]]})

resumen_final = pd.DataFrame([
    ("Fichas nominales recibidas", len(nominal_records_todos), ""),
    ("Fichas excluidas por código inconsistente", len(nominal_excluida), "01_ids_inconsistentes_excluidos.xlsx"),
    ("Evaluaciones de nutrición recibidas", len(nutrition_df_bruto), ""),
    ("Evaluaciones excluidas por código inconsistente", len(evaluaciones_ids_excluidas), "01_ids_inconsistentes_excluidos.xlsx"),
    ("Evaluaciones excluidas por PROVINCIA", len(evaluaciones_provincia), "02_evaluaciones_provincia_excluidas.xlsx"),
    ("Celdas de diagnóstico con texto normalizado", int((dx_largo["tipo_cambio"] != "sin_cambio").sum()), "03_normalizacion_texto_aplicada.xlsx"),
    ("IMC completados por nosotros", int(nutrition_df["imc_fue_calculado_por_nosotros"].sum()), "04_imc_completado.xlsx"),
    ("Diagnósticos INS calculados por nosotros", len(calculos), "05_diagnosticos_calculados.xlsx"),
    ("Filas del consolidado final", len(unified_df), "Resultados/Consolidado/Dataset_unificado.xlsx"),
    ("Columnas del consolidado final", unified_df.shape[1], ""),
], columns=["concepto", "cantidad", "evidencia"])
show_table(resumen_final.set_index("concepto"), "Resumen final del Paso 3")
export_evidence("00_resumen_paso3.xlsx", {"resumen": resumen_final})

Evidencia guardada: C - KDD Steps/C - 3: Data Cleaning and Preprocessing/08_nutricion_sin_union_con_ficha_nominal.xlsx  (resumen: 2 filas, id_ausente_en_mes_nominal: 5 filas, periodo_nominal_no_disponible: 32 filas)
Evidencia guardada: C - KDD Steps/C - 3: Data Cleaning and Preprocessing/09_fichas_sin_nutricion_por_provincia.xlsx  (fichas_nominales_mes_sin_evaluacion: 23 filas)


,cantidad,evidencia
concepto,,
Fichas nominales recibidas,2667,
Fichas excluidas por código inconsistente,169,01_ids_inconsistentes_excluidos.xlsx
Evaluaciones de nutrición recibidas,402,
Evaluaciones excluidas por código inconsistente,24,01_ids_inconsistentes_excluidos.xlsx
Evaluaciones excluidas por PROVINCIA,23,02_evaluaciones_provincia_excluidas.xlsx
Celdas de diagnóstico con texto normalizado,703,03_normalizacion_texto_aplicada.xlsx
IMC completados por nosotros,6,04_imc_completado.xlsx
Diagnósticos INS calculados por nosotros,3,05_diagnosticos_calculados.xlsx
Filas del consolidado final,2535,Resultados/Consolidado/Dataset_unificado.xlsx


Evidencia guardada: C - KDD Steps/C - 3: Data Cleaning and Preprocessing/00_resumen_paso3.xlsx  (resumen: 10 filas)


In [ ]:
# RESUMEN AMPLIADO: se agregan las cifras de los tratamientos 6 y 7, de la evaluación de vacíos y del consolidado 2
resumen_final = pd.concat([resumen_final, pd.DataFrame([
    ("IMC corregidos (el calculado reemplaza al reportado por diferencia > 0.11)", int(nutrition_df["imc_corregido_por_diferencia"].sum()), "11_imc_corregido_por_diferencia.xlsx"),
    ("Columnas de la ficha nominal limpiadas (comillas, fechas, texto)", df_nominal_limpio.shape[1], "10_limpieza_ficha_nominal.xlsx"),
    ("Celdas de texto de la ficha nominal homologadas (fichas que continúan)", int(resumen_limpieza["celdas_homologadas"].sum()), "10_limpieza_ficha_nominal.xlsx"),
    ("Columnas evaluadas por vacíos (nutrición + ficha nominal)", len(evaluacion_vacios), "12_evaluacion_vacios.xlsx"),
    ("Columnas que no pasan al consolidado 2", int(evaluacion_vacios["decision"].str.startswith("ELIMINAR").sum()), "12_evaluacion_vacios.xlsx"),
    ("Filas del consolidado 2", len(unified_2), "Resultados/Consolidado/Dataset_unificado_2.xlsx"),
    ("Columnas del consolidado 2", unified_2.shape[1], "13_vacios_consolidado_2.xlsx"),
], columns=resumen_final.columns)], ignore_index=True)
resumen_final.loc[resumen_final["concepto"] == "Columnas del consolidado final", "cantidad"] = unified_df.shape[1]
show_table(resumen_final.set_index("concepto"), "Resumen final del Paso 3 (ampliado)")
export_evidence("00_resumen_paso3.xlsx", {"resumen": resumen_final})

,cantidad,evidencia
concepto,,
Fichas nominales recibidas,2667,
Fichas excluidas por código inconsistente,169,01_ids_inconsistentes_excluidos.xlsx
Evaluaciones de nutrición recibidas,402,
Evaluaciones excluidas por código inconsistente,24,01_ids_inconsistentes_excluidos.xlsx
Evaluaciones excluidas por PROVINCIA,23,02_evaluaciones_provincia_excluidas.xlsx
Celdas de diagnóstico con texto normalizado,703,03_normalizacion_texto_aplicada.xlsx
IMC completados por nosotros,6,04_imc_completado.xlsx
Diagnósticos INS calculados por nosotros,3,05_diagnosticos_calculados.xlsx
Filas del consolidado final,2535,Resultados/Consolidado/Dataset_unificado.xlsx


Evidencia guardada: C - KDD Steps/C - 3: Data Cleaning and Preprocessing/00_resumen_paso3.xlsx  (resumen: 17 filas)


**Comentario 21:** De 2 667 fichas y 402 evaluaciones recibidas, el consolidado conserva 2 498 fichas y 355 evaluaciones: se excluyeron 169 fichas y 24 evaluaciones por códigos inconsistentes y 23 evaluaciones por `PROVINCIA`, se normalizaron 703 celdas de diagnóstico, se completaron 6 IMC y se calcularon 3 diagnósticos con tablas INS. Todas las separaciones de datos están en `Resultados/Paso3_evidencias/`:

| Archivo | Contenido |
|---|---|
| `01_ids_inconsistentes_excluidos.xlsx` | Resumen por código, fichas nominales y evaluaciones excluidas |
| `02_evaluaciones_provincia_excluidas.xlsx` | Evaluaciones sin medición por `PROVINCIA` |
| `03_normalizacion_texto_aplicada.xlsx` | Variantes de texto y su forma normalizada |
| `04_imc_completado.xlsx` | IMC calculados por nosotros |
| `05_diagnosticos_calculados.xlsx` | Diagnósticos calculados, con el detalle y la sensibilidad a la edad |
| `06_asteriscos_que_se_conservan.xlsx` | Asteriscos del bloque INS que no se calcularon y su motivo |
| `07_estado_final_asteriscos.xlsx` | Estado final de todos los asteriscos |
| `08_nutricion_sin_union_con_ficha_nominal.xlsx` | Evaluaciones que no se unieron, por motivo |
| `09_fichas_sin_nutricion_por_provincia.xlsx` | Fichas de meses sin evaluación por `PROVINCIA` |
| `00_resumen_paso3.xlsx` | Resumen con las cifras de este cuaderno |


**Comentario 21A:** Además de la lista anterior, se guardan en `Resultados/Paso3_evidencias/`:

| Archivo | Contenido |
|---|---|
| `10_limpieza_ficha_nominal.xlsx` | Limpieza de la ficha nominal: resumen por columna, formatos de fecha, fechas de dos dígitos, cambios de texto y muestra antes/después |
| `11_imc_corregido_por_diferencia.xlsx` | NNA con IMC corregido: resumen y fila completa de la nutrición antes y después |
| `12_evaluacion_vacios.xlsx` | Decisión por columna, diagnósticos vacíos donde sí aplican, cobertura por mes y evaluaciones por NNA |
| `13_vacios_consolidado_2.xlsx` | Vacíos que quedan en el consolidado 2 y su lista de columnas |

Y en `Resultados/Consolidado/`: `Dataset_unificado.xlsx` (consolidado 1, 168 columnas), `Nutricion_consolidada.xlsx` (nutrición tratada, con las columnas nuevas de este tramo) y **`Dataset_unificado_2.xlsx`** (consolidado 2, solo valores tratados).